In [ ]:
from astropy.io import fits
from astropy.coordinates import SkyCoord
from astropy.wcs import WCS
from astropy.time import Time
import skyloc
import astropy.units as u

from astropy.nddata import Cutout2D

from pathlib import Path
import pandas as pd
import numpy as np
import sep
from skimage.draw import disk

import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from astropy.visualization import ZScaleInterval
import rcparams
import directory

from tqdm import tqdm
import re

In [ ]:
FITS_DIR   = directory.FITS_DIR
DB_DIR     = directory.DB_DIR
FIG_DIR    = directory.FIG_DIR
REFCAT_DIR = directory.REFCAT_DIR # Gaia DR3 catalog directory

In [ ]:
objdesig = "2014 UN271"
fpath_db = DB_DIR / "db_filtered.parq"

# Use pushdown filtering to only load memory for 2P
data_summary = pd.read_parquet(
    fpath_db, 
    filters=[("objdesig", "==", objdesig)]
)

# Inspect the filtered dataframe
data_summary.info()
data_summary

## Update additional metadata for data_summary

#### WCS_SERIALIZED, RA-OBJ, DEC-OBJ from Fits Header

In [ ]:
for row in tqdm(data_summary.itertuples(), total=len(data_summary), desc="Extracting WCS"):
    
    fpath_fits = FITS_DIR / row.filename
    hdr = fits.getheader(fpath_fits, ext=1)
    
    ra_obj = hdr['RA-OBJ']
    dec_obj = hdr['DEC-OBJ']
    wcs_serialized = WCS(hdr).to_header_string(relax=True)
    
    data_summary.loc[row.Index, 'ra_obj'] = ra_obj
    data_summary.loc[row.Index, 'dec_obj'] = dec_obj
    data_summary.loc[row.Index, 'wcs_serialized'] = wcs_serialized
    

# 1. Define Regex patterns to elegantly catch all standard WCS and Distortion keywords
wcs_patterns = [
    r"^RADESYS$", r"^WCSAXES$",
    r"^CRPIX\d$", r"^CRVAL\d$", r"^CTYPE\d$", r"^CUNIT\d$", r"^CDELT\d$",
    r"^PC\d_\d$", r"^LONPOLE$", r"^LATPOLE$",
    r"^PV\d_\d+$",                     # PV Distortion coefficients
    r"^[AB]_ORDER$", r"^[AB]_\d_\d$",  # SIP Forward Distortion (A/B)
    r"^[AB]P_ORDER$", r"^[AB]P_\d_\d$" # SIP Reverse Distortion (AP/BP)
]

# 2. Extract the exact column names from your dataframe that match these patterns
wcs_cols = [
    col for col in data_summary.columns 
    if any(re.match(pattern, col) for pattern in wcs_patterns)
]
data_summary = data_summary.drop(columns=wcs_cols)

data_summary.info()
data_summary.head()

#### Phase

In [ ]:
def spherex_phase_grouping(fits_summary, threshold_days=28, key_dateobs='DATE-OBS'):
    """
    Groups FITS file records into discrete observation phases based on the time gap 
    between consecutive observations.

    This function sorts the input DataFrame by the specified observation date column. 
    It then calculates the time difference between consecutive rows and assigns a new 
    phase number whenever the gap exceeds the defined `threshold_days`.

    Args:
        fits_summary (pandas.DataFrame): A DataFrame containing summary information 
            of FITS files, which must include a column with observation dates.
        threshold_days (int or float, optional): The maximum allowable time gap (in days) 
            between consecutive observations to remain in the same phase. A gap strictly 
            greater than this value triggers a new phase. Defaults to 28.
        key_dateobs (str, optional): The column name in `fits_summary` that contains 
            the observation datetimes. Defaults to 'DATE-OBS'.

    Returns:
        pandas.DataFrame: A copy of the input DataFrame, sorted chronologically by 
            `key_dateobs`, with an additional integer column named 'PHASE' (starting 
            from 1) indicating the grouped phase of each observation.
    """
    
    fits_summary_updated = fits_summary.copy()
    
    dateobs = pd.to_datetime(fits_summary_updated[key_dateobs])
    
    # Sort by dateobs to ensure correct phase grouping
    sort_idx = dateobs.argsort()
    fits_summary_updated = fits_summary_updated.iloc[sort_idx].reset_index(drop=True)
    dateobs = dateobs.iloc[sort_idx].reset_index(drop=True)
    
    time_diff = dateobs.diff()
    threshold = pd.Timedelta(days=threshold_days)
    is_new_phase = time_diff > threshold
    
    fits_summary_updated['phase'] = is_new_phase.cumsum() + 1
    
    return fits_summary_updated

data_summary = spherex_phase_grouping(data_summary, threshold_days=28, key_dateobs='DATE-OBS')
data_summary.value_counts('phase').sort_index()

In [ ]:
def read_npy_subset(fpath, columns):
    """
    Reads specific columns from a structured .npy file while minimizing memory usage.

    Args:
        fpath (str or Path): Path to the .npy file.
        columns (list of str): List of column names to extract.

    Returns:
        np.ndarray: A contiguous numpy array in memory containing only the requested columns.
    """
    
    mmap_data = np.load(fpath, mmap_mode='r')
    
    # Optional safety check: ensure all requested columns exist
    available_cols = mmap_data.dtype.names
    for col in columns:
        if col not in available_cols:
            raise ValueError(f"Column '{col}' not found. Available columns: {available_cols}")
    
    subset_view = mmap_data[columns]
    subset_data = np.array(subset_view, copy=True)
    
    del mmap_data
    
    return subset_data

fpath_gaia = REFCAT_DIR / "gaiadr3.npy"
gaia_all = read_npy_subset(fpath_gaia, columns=['ra', 'dec', 'phot_g_mean_mag'])
gaia_all

In [ ]:
from typing import Any

def create_gaia_subset(
    gaia_all: Any, 
    fits_summary: pd.DataFrame, 
    del_ra_deg: float, 
    del_dec_deg: float, 
    gmag_limit: float, 
    ra_col: str = 'ra', 
    dec_col: str = 'dec'
) -> Any:
    """
    Filters a massive Gaia catalog for multiple overlapping target regions.
    Optimized via magnitude cuts, global bounding boxes, unique pointings, 
    and dynamic spherical RA adjustments.
    """
    # ---------------------------------------------------------
    # STEP 1: Global Pre-filtering (The fastest reductions)
    # ---------------------------------------------------------
    # Start with the magnitude limit
    global_mask = gaia_all['phot_g_mean_mag'] < gmag_limit
    
    # Isolate valid coordinates and drop NaNs
    valid_coords = fits_summary[[ra_col, dec_col]].dropna()
    if valid_coords.empty:
        print("Warning: No valid RA/DEC coordinates found in fits_summary.")
        return gaia_all[0:0] # Return empty array
        
    # Global DEC bounds (throws away useless hemispheres)
    dec_min_global = valid_coords[dec_col].min() - del_dec_deg
    dec_max_global = valid_coords[dec_col].max() + del_dec_deg
    global_mask &= (gaia_all['dec'] >= dec_min_global) & (gaia_all['dec'] <= dec_max_global)
    
    # Apply global pre-filters to create a much smaller working array
    gaia_filtered = gaia_all[global_mask]
    
    # Extract columns to 1D arrays for fast vectorized operations
    ra_gaia = gaia_filtered['ra']
    dec_gaia = gaia_filtered['dec']
    
    # ---------------------------------------------------------
    # STEP 2: Exact Spatial Masking (Dynamic Grid Grouping)
    # ---------------------------------------------------------
    exact_spatial_mask = np.zeros(len(gaia_filtered), dtype=bool)
    
    # 1. Determine grid resolution (50% of the smaller delta)
    grid_res = min(del_ra_deg, del_dec_deg) * 0.5
    
    # 2. Snap coordinates to the nearest grid_res interval and drop duplicates
    unique_pointings = (valid_coords / grid_res).round() * grid_res
    unique_pointings = unique_pointings.drop_duplicates()
    
    # 3. Base effective deltas (compensating for grid snapping)
    eff_del_ra_base = del_ra_deg + (grid_res / 2.0)
    eff_del_dec = del_dec_deg + (grid_res / 2.0)
    
    # Use itertuples for massive speedup over iterrows
    for ra_obj, dec_obj in unique_pointings.itertuples(index=False):
        
        # Local DEC mask 
        local_dec_mask = (dec_gaia >= dec_obj - eff_del_dec) & (dec_gaia <= dec_obj + eff_del_dec)
        
        # Adjust RA delta based on Declination (Spherical geometry projection)
        if abs(dec_obj) > 89.0:
            # If right at the pole, grab all RA values
            eff_del_ra_local = 180.0 
        else:
            cos_dec = np.cos(np.radians(dec_obj))
            eff_del_ra_local = min(180.0, eff_del_ra_base / cos_dec)
            
        # Local RA mask safely handling 0/360 degree wraparound
        ra_min = (ra_obj - eff_del_ra_local) % 360
        ra_max = (ra_obj + eff_del_ra_local) % 360
        
        if ra_min < ra_max:
            local_ra_mask = (ra_gaia >= ra_min) & (ra_gaia <= ra_max)
        else:
            # Wraparound condition
            local_ra_mask = (ra_gaia >= ra_min) | (ra_gaia <= ra_max)
            
        # Combine local masks
        exact_spatial_mask |= (local_ra_mask & local_dec_mask)
        
    # ---------------------------------------------------------
    # STEP 3: Return Final Subset
    # ---------------------------------------------------------
    return gaia_filtered[exact_spatial_mask]

In [ ]:
gaia_subset_phase = dict()
del_ra_deg, del_dec_deg = 0.2, 0.2
gmag_limit = 18

for phase in data_summary['phase'].unique():
    phase_mask = data_summary['phase'] == phase
    data_summary_phase = data_summary[phase_mask]
    
    gaia_subset = create_gaia_subset(
        gaia_all, data_summary_phase, 
        del_ra_deg=del_ra_deg, del_dec_deg=del_dec_deg, gmag_limit=gmag_limit,
        ra_col='ra', dec_col='dec'
    )
    gaia_subset_phase[phase] = gaia_subset
    
    del gaia_subset # remove reference to free memory

In [ ]:
num_phases = len(gaia_subset_phase)
fig = plt.figure(figsize=(8*num_phases, 8))
gs = GridSpec(1, num_phases, figure=fig)

for i, (phase, gaia_subset) in enumerate(gaia_subset_phase.items()):

    
    ax = fig.add_subplot(gs[i])
    
    # Plot Gaia Background
    ax.scatter(
        gaia_subset['ra'], gaia_subset['dec'], 
        s=10, alpha=0.5, color='green', 
        label=f'Gaia Source ($G<{gmag_limit}$)' if i == 0 else None
    )
    data_summary_gp_phase = data_summary[data_summary['phase'] == phase]

    # SPEED OPTIMIZATION: Use itertuples()
    for row in data_summary_gp_phase.itertuples():
        
        header = fits.Header.fromstring(row.wcs_serialized)
        # header = fits.getheader(FITS_DIR / row.filename, ext=1)
        wcs = WCS(header)
        
        ny, nx = row.cutout_size, row.cutout_size  
        ra_obj, dec_obj = row.ra_obj, row.dec_obj
        
        # Plot Target Center
        ax.scatter(
            ra_obj, dec_obj, s=50, color='yellow', edgecolor='black', marker='*',
            label='Target' if row.Index == data_summary_gp_phase.index[0] else None
        )
        
        # Calculate Footprint Corners
        x_corners = [0, nx, nx,  0, 0]
        y_corners = [0,  0, ny, ny, 0]
        sky_corners = wcs.pixel_to_world(x_corners, y_corners)
        
        ra_poly = sky_corners.ra.deg
        dec_poly = sky_corners.dec.deg
        
        # GEOMETRY FIX: Prevent RA Wrap-Around Streaks
        # If the box spans more than 180 degrees, it has crossed the 0/360 boundary.
        if np.max(ra_poly) - np.min(ra_poly) > 180.0:
            # Shift the high RA values to negative numbers so Matplotlib draws a contiguous box
            ra_poly = np.where(ra_poly > 180.0, ra_poly - 360.0, ra_poly)
            
        # Plot the bounding box
        label = 'SPHEREx Cutout FoV' if row.Index == data_summary_gp_phase.index[0] else None
        ax.plot(ra_poly, dec_poly, color='red', linewidth=1.5, alpha=0.2, label=label)
        
    # Formatting
    date_obs_min = Time(data_summary_gp_phase['DATE-OBS'].min()).to_datetime().strftime('%Y-%m-%d')
    date_obs_max = Time(data_summary_gp_phase['DATE-OBS'].max()).to_datetime().strftime('%Y-%m-%d')
    
    ax.set_title(f"Phase {phase}\n({date_obs_min} to {date_obs_max})")
    ax.grid(True, linestyle='--', alpha=0.3)
    ax.set_xlabel(r'$\alpha~(\degree)$')
    ax.set_ylabel(r'$\delta~(\degree)$')
    
    if i == 0:
        ax.legend()

plt.tight_layout() # Ensure titles and axes don't overlap
plt.suptitle(f"{objdesig}", y=1.02)  # Adjust y to place the title above the subplots
# plt.savefig(FIG_DIR / 'spherex_gaia_coverage_phases.png')
plt.show()

In [ ]:
def flag_to_mask(flag, flag_number):
    mask = np.zeros_like(flag, dtype=bool)
    for f in flag_number:
        mask |= (flag & (1 << f)) != 0
    return mask.astype(bool)

def gaia_to_mask(gaia_subset, sci, gmag_limit, mag_col='phot_g_mean_mag', radius_scale=1.0):
    """
    Generates a boolean mask array where circular regions around Gaia sources are masked (True).
    The radius of each mask depends on the star's magnitude and the image's PSF FWHM.

    Args:
        gaia_subset (numpy.ndarray or pandas.DataFrame): The filtered catalog of Gaia sources.
        wcs (astropy.wcs.WCS): The World Coordinate System of the target image.
        shape (tuple): A tuple of (ny, nx) representing the 2D dimensions of the image.
        psf_fwhm (float): The Full Width at Half Maximum of the PSF for the specific image.
        gmag_limit (float): The limiting magnitude. Fainter stars will have smaller 
            (or zero) mask radii.
        ra_col (str, optional): The column name for Right Ascension. Defaults to 'ra'.
        dec_col (str, optional): The column name for Declination. Defaults to 'dec'.
        mag_col (str, optional): The column name for G-band magnitude. Defaults to 'phot_g_mean_mag'.
        radius_scale (float, optional): The scaling factor for the mask radius. Defaults to 0.3.

    Returns:
        numpy.ndarray: A 2D boolean array of shape (ny, nx) where True indicates a masked star pixel.
    """
    
    hdr = sci.header
    ny, nx = hdr['NAXIS2'], hdr['NAXIS1']
    wcs = WCS(hdr)
    psf_fwhm_arcsec = hdr.get('PSF_FWHM', 6.0) # Default to 6 arcsec if not available
    psf_fwhm_pixel = psf_fwhm_arcsec / hdr.get('PIX-SCL', 6.0) # Convert FWHM to pixels using pixel scale
    
    mask_source = np.zeros((ny, nx), dtype=bool)
    
    # Extract coordinates and magnitudes
    ra_gaia   = np.array(gaia_subset['ra'], dtype=float)
    dec_gaia  = np.array(gaia_subset['dec'], dtype=float)
    gmag_gaia = np.array(gaia_subset[mag_col], dtype=float)
    
    # Convert sky coordinates to pixel coordinates
    skycoords_gaia = SkyCoord(ra=ra_gaia*u.deg, dec=dec_gaia*u.deg)
    x_gaia, y_gaia = wcs.world_to_pixel(skycoords_gaia)
    
    # Draw circular masks
    for xc, yc, gmag in zip(x_gaia, y_gaia, gmag_gaia):
        # Skip unprojectable coordinates
        if np.isnan(xc) or np.isnan(yc):
            continue
        
        # Calculate dynamic radius based on brightness
        radius = radius_scale * psf_fwhm_pixel * (gmag_limit - gmag)
        
        # Skip if the star is fainter than the limit (radius <= 0)
        if radius <= 0:
            continue
        
        # Get pixel indices for the circular disk
        # `shape=(ny, nx)` ensures it automatically clips at the image boundaries
        rr, cc = disk((yc, xc), radius, shape=(ny, nx))
        
        # Apply the mask
        mask_source[rr, cc] = True
        
    return mask_source

In [ ]:
# cutout example
row = data_summary.iloc[100]

hdul = fits.open(FITS_DIR / row['filename'])
sci  = hdul[1]
var  = hdul[2] # variance (error^2) map
flag = hdul[3] # flag bitmap
phase = row['phase']
hdul.info()

In [ ]:
#### Mask SPHEREx flag
flag_number = [2, 6, 7, 9, 10, 11, 12, 14, 15, 17, 22, 24, 26, 27, 28, 29] # exclude: 19, 21
# 0: transient (e.g., cosmic ray)
# 1: overflow (half-saturated)
# 2: sur_error
# 6: permanently dead pixel
# 7: smile effect (band 3 and 4)
# 9: missing data
# 10: hot pixel
# 11: cold pixel
# 12: fullsample
# 14: phantom missing data
# 15: non-linear
# 17: affected by persistent charge
# 19: outlier pixel ==> likely transient
# 21: known source
# 22&24: affected by optical ghost
# 26: affected by source "blooming"
# 27: affected by "snowball" events
# 28: affected by "halo", especially around cosmic
# 29: affected by satellite streak
mask_flag = flag_to_mask(flag.data, flag_number)
print(f"Total flagged pixels: {np.sum(mask_flag)}/{mask_flag.size}")

In [ ]:
gaia_subset = gaia_subset_phase[phase]
mask_source = gaia_to_mask(gaia_subset, sci, gmag_limit, mag_col='phot_g_mean_mag', radius_scale=1.0)
print(f"Total Gaia-masked pixels: {np.sum(mask_source)}/{mask_source.size}")
# mask_source = flag_to_mask(flag.data, flag_number=[21])

In [ ]:
sci_masked = sci.data.copy()
sci_masked[mask_flag | mask_source] = np.nan
print(f"Total masked pixels: {np.isnan(sci_masked).sum()}/{sci_masked.size}")

In [ ]:
fig = plt.figure(figsize=(12, 4))
gs = GridSpec(1, 3, figure=fig)

ax = fig.add_subplot(gs[0, 0])
vmin, vmax = ZScaleInterval().get_limits(sci_masked) # Use only unmasked pixels for scaling
ax.imshow(sci.data, origin='lower', cmap='magma', vmin=vmin, vmax=vmax)
ax.scatter(row['xcen'], row['ycen'], s=100, color='b', edgecolor='b', marker='x', label=objdesig)
ax.set_xlabel('X [pix]')
ax.set_ylabel('Y [pix]')
# ax.set_xlim(row['X-OBJ']-50, row['X-OBJ']+50)
# ax.set_ylim(row['Y-OBJ']-50, row['Y-OBJ']+50)
ax.set_title("Original")

ax = fig.add_subplot(gs[0, 1])
ax.imshow(mask_flag | mask_source, cmap='gray')
# ax.set_xlim(row['X-OBJ']-50, row['X-OBJ']+50)
# ax.set_ylim(row['Y-OBJ']-50, row['Y-OBJ']+50)
# ax.set_title(f"Gaia (G<{gmag_limit})+ Flag")
ax.invert_yaxis()  # Invert y-axis to match image orientation
ax.set_title(f"Mask\n($G<{gmag_limit}$ && Data Flag)")

ax = fig.add_subplot(gs[0, 2])
ax.imshow(sci_masked, origin='lower', cmap='magma', vmin=vmin, vmax=vmax)
# ax.set_xlim(row['X-OBJ']-50, row['X-OBJ']+50)
# ax.set_ylim(row['Y-OBJ']-50, row['Y-OBJ']+50)
ax.set_title("Data Masked")
ax.scatter(row['xcen'], row['ycen'], s=100, color='b', edgecolor='b', marker='x', label=objdesig)

plt.suptitle(row['filename'], y=0, fontsize=12)  # Adjust y to place the title above the subplots
# plt.savefig(FIG_DIR / f"cutout_masked_{row['FILENAME'].replace('.fits', '.png')}")
plt.show()

# 2. Aperture photometry

In [ ]:
import astropy.units as u
from tqdm import tqdm

from photutils.aperture import CircularAperture, CircularAnnulus, aperture_photometry
from astropy.stats import sigma_clip

data_summary["RHO_KM"] = 20000 # projected radius km

pixel_scale_km = (data_summary["PIX-SCL"] * ((1*u.arcsec).to(u.rad).value) * data_summary["R_OBS"] * (1*u.au).to(u.km).value) # km/pixel
data_summary["RHO_PIX"] = data_summary["RHO_KM"] / pixel_scale_km

# size of background annulus
data_summary["AN_IN_PIX"]  = 6*data_summary["RHO_PIX"]
data_summary["AN_OUT_PIX"] = 8*data_summary["RHO_PIX"]
# data_summary["rho_fwhm"] = data_summary["RHO_PIX"] / data_summary["FWHM_PIX"]

In [ ]:
def comet_aperture_photometry(fits_summary, fits_dir, gaia_subset_phased, gmag_limit, sci_ext=1, err_ext=2, flag_ext=3,
                              x_col='X-OBJ', y_col='Y-OBJ', flux_col='AP-SUM1', error_col='AP-ERR1', neff_col='AP-NEFF1'):
    """
    Performs robust aperture photometry on cometary targets.
    
    Improvements include:
    1. Robust background estimation (sigma-clipped median).
    2. Effective area calculation (ignoring NaN/masked pixels inside the aperture).
    3. Rigorous DAOPHOT error propagation.
    4. Safe handling of NaN values, pipeline flags, and Gaia source masks.
    """
    
    fits_summary_updated = fits_summary.copy()
    
    # Initialize output columns
    fits_summary_updated[flux_col] = np.nan
    fits_summary_updated[error_col] = np.nan
    fits_summary_updated[neff_col] = np.nan
    
    # Convert fits_dir to Path object if it isn't already
    fits_dir = Path(fits_dir)
    
    for idx, row in tqdm(fits_summary_updated.iterrows(), total=len(fits_summary_updated), desc="Aperture Photometry"):
        
        fpath = fits_dir / row['FILENAME']
        
        try:
            with fits.open(fpath) as hdul:
                sci = hdul[sci_ext]
                sci_data = sci.data.astype(np.float32)
                
                # Try to load error array if specified and available
                if err_ext is not None and len(hdul) > err_ext:
                    err_data = np.sqrt(hdul[err_ext].data.astype(np.float32))
                else:
                    err_data = None
                    
                # Try to load flag array if specified and available
                if flag_ext is not None and len(hdul) > flag_ext:
                    flag_data = hdul[flag_ext].data.astype(np.uint32)
                    mask_flag = flag_to_mask(flag_data, flag_number=[0, 2, 6, 7, 9, 10, 11, 12, 14, 15, 17, 22, 24, 26, 27, 28, 29])
                else:
                    mask_flag = np.zeros_like(sci_data, dtype=bool)
                
                # Generate Gaia mask
                gaia_subset = gaia_subset_phased.get(row['PHASE'], pd.DataFrame()) # Get the Gaia subset for this phase
                mask_source = gaia_to_mask(gaia_subset, sci, gmag_limit, mag_col='phot_g_mean_mag', radius_scale=1.0)
                
                # Combine flags, Gaia sources, and intrinsic NaNs into a MASTER MASK
                # (True means the pixel is bad/masked and should be ignored)
                nan_mask = np.isnan(sci_data)
                master_mask = mask_flag | mask_source | nan_mask
                
                xcen, ycen = row[x_col], row[y_col]
                if np.isnan(xcen) or np.isnan(ycen):
                    raise ValueError(f"Invalid center coordinates ({xcen}, {ycen}).")
                
                position = (xcen, ycen)
                rho_pix = row["RHO_PIX"]
                r_in = row["AN_IN_PIX"]
                r_out = row["AN_OUT_PIX"]
                
                # 1. Define Aperture & Annulus
                aperture = CircularAperture(position, r=rho_pix)
                annulus = CircularAnnulus(position, r_in=r_in, r_out=r_out)
                
                # 3. Estimate Sky Background (Sigma-Clipped Median)
                ann_mask = annulus.to_mask(method='center')
                ann_cutout = ann_mask.cutout(sci_data)
                ann_master_cutout = ann_mask.cutout(master_mask) # Cut out the master mask
                
                if ann_cutout is None or ann_master_cutout is None:
                    raise ValueError("Annulus falls entirely outside the image.")
                    
                # Extract only pixels inside the annulus that are NOT in the master mask
                valid_sky_pixels = ann_cutout[(ann_mask.data > 0) & (~ann_master_cutout)]
                
                if len(valid_sky_pixels) < 10:
                    raise ValueError("Not enough valid sky pixels to estimate background.")
                    
                sky_clipped = sigma_clip(valid_sky_pixels, sigma=3.0, maxiters=5)
                msky = np.ma.median(sky_clipped)  # Median sky per pixel
                ssky = np.ma.std(sky_clipped)     # Sky standard deviation
                nsky = sky_clipped.count()        # Number of unmasked sky pixels used
                
                # 4. Calculate Effective Aperture Area
                ap_mask = aperture.to_mask(method='center')
                ap_master_cutout = ap_mask.cutout(master_mask) # Cut out the master mask
                
                if ap_master_cutout is None:
                    raise ValueError("Aperture falls entirely outside the image.")
                
                # Count valid pixels inside the aperture (ignore flagged, Gaia, and NaN pixels)
                eff_area = np.sum((ap_mask.data > 0) & (~ap_master_cutout))
                
                
                # 5. Perform Raw Aperture Photometry
                # Pass the master_mask so photutils ignores bad pixels when summing the flux
                phot_table = aperture_photometry(sci_data, aperture, error=err_data, mask=master_mask)
                raw_flux = phot_table['aperture_sum'][0]
                
                # 6. Correct Flux and Calculate Error
                net_flux = raw_flux - (msky * eff_area)
                
                # Rigorous DAOPHOT Error Formula
                if err_data is not None:
                    poisson_err = phot_table['aperture_sum_err'][0]
                else:
                    poisson_err = np.sqrt(abs(net_flux))
                
                total_error = np.sqrt(
                    poisson_err**2 + 
                    (eff_area * ssky**2) + 
                    ((eff_area**2 * ssky**2) / nsky)
                )
                
                # Store results
                fits_summary_updated.at[idx, flux_col] = net_flux
                fits_summary_updated.at[idx, error_col] = total_error
                fits_summary_updated.at[idx, neff_col] = eff_area
                
        except Exception as e:
            tqdm.write(f"Skipped {row.get('FILENAME', 'Unknown')}: {e}")
            
    return fits_summary_updated

In [ ]:
fig = plt.figure(figsize=(15, 3*(len(fits_summary)//5+1)))
gs  = GridSpec(nrows=len(fits_summary)//5+1, ncols=5, figure=fig)

for idx, row in tqdm(fits_summary.iterrows(), total=len(fits_summary), desc="Performing aperture photometry"):
    
    filepath = FITS_DIR / row['FILENAME']
    # with fits.open(filepath) as hdul:
    hdul = fits.open(filepath)
    sci = hdul[1]
    wcs_header = fits.Header.fromstring(row['WCS_SERIALIZED'])
    wcs  = WCS(wcs_header)
    err  = np.sqrt(hdul[2].data.astype(np.float32)) # error array (variance^0.5)
    flag = hdul[3] # flag bitmap
    phase = row['PHASE'] # phase grouping for Gaia subset selection

    # Apply masks
    mask_flag = flag_to_mask(flag.data, flag_number)
    gaia_subset = gaia_subset_phase[phase]
    mask_source = gaia_to_mask(gaia_subset, sci, gmag_limit, mag_col='phot_g_mean_mag', radius_scale=1.0)
    overall_mask = (mask_flag | mask_source)
    
    sci_masked = sci.data.copy()
    sci_masked[overall_mask] = np.nan
    
    # skycoord = SkyCoord(ra=row.ra*u.deg, dec=row.dec*u.deg, frame='icrs')
    # xycoord = wcs.world_to_pixel(skycoord)

    # xycen_obj = np.array([row['XCEN'], row['YCEN']])
    # xycen_frag = np.array([row['XCEN-FRAG'], row['YCEN-FRAG']])

    # Refine object position with SEP.winpos
    # xycoord_winpos = sep.winpos(data, xinit=xycen_obj[0], yinit=xycen_obj[1], sig=3*row.fwhm_pix)

    # Define apertures
    ap_obj = CircularAperture((row['X-OBJ'], row['Y-OBJ']), r=row['RHO_PIX'])
    an_obj  = CircularAnnulus((row['X-OBJ'], row['Y-OBJ']), r_in=row['AN_IN_PIX'], r_out=row['AN_OUT_PIX'])

    ap_frag = CircularAperture((row['X-FRAG'], row['Y-FRAG']), r=row['RHO_PIX'])
    an_frag = CircularAnnulus((row['X-FRAG'], row['Y-FRAG']), r_in=row['AN_IN_PIX'], r_out=row['AN_OUT_PIX'])

    # Get aperture masks
    ap_mask_obj = ap_obj.to_mask(method='center')
    ap_data_cutout = ap_mask_obj.cutout(sci.data)
    ap_mask_cutout = ap_mask_obj.cutout(overall_mask) # Get the boolean mask for the same cutout
    
    ap_mask_frag = ap_frag.to_mask(method='center')
    ap_data_cutout_frag = ap_mask_frag.cutout(sci.data)
    ap_mask_cutout_frag = ap_mask_frag.cutout(overall_mask)
    
    # Calculate effective area (pixels in aperture that are NOT masked)
    valid_ap_pixels = (ap_mask_obj.data > 0) & (~ap_mask_cutout)
    eff_area_obj = np.sum(valid_ap_pixels)
    
    valid_ap_pixels_frag = (ap_mask_frag.data > 0) & (~ap_mask_cutout_frag)
    eff_area_frag = np.sum(valid_ap_pixels_frag)
    
    # Perform aperture photometry
    phot_obj  = aperture_photometry(sci.data, ap_obj, error=err, mask=overall_mask)
    phot_frag = aperture_photometry(sci.data, ap_frag, error=err, mask=overall_mask)

    # Estimate sky background from annulus
    mask_obj  = an_obj.to_mask(method='center')
    ann_cutout = mask_obj.cutout(sci.data)  
    ann_mask_cutout = mask_obj.cutout(overall_mask)
    
    mask_frag = an_frag.to_mask(method='center')
    ann_cutout_frag = mask_frag.cutout(sci.data)
    ann_mask_cutout_frag = mask_frag.cutout(overall_mask)
    
    ## Object photometry
    if ann_cutout is None:
        msky = np.nan
        ssky = np.nan
        nsky = 0
    else:
        # Extract pixels that are inside the annulus AND NOT masked by Gaia/Flags
        valid_sky_mask = (mask_obj.data > 0) & (~ann_mask_cutout)
        sky_data = ann_cutout[valid_sky_mask]
        
        # Sigma clip the valid sky pixels
        sky_data_clipped = sigma_clip(sky_data, sigma=3, maxiters=10)
        msky = np.ma.median(sky_data_clipped)
        ssky = np.ma.std(sky_data_clipped)
        nsky = sky_data_clipped.count()
        
    if ann_cutout_frag is None:
        msky_frag = np.nan
        ssky_frag = np.nan
        nsky_frag = 0
    else:
        valid_sky_mask_frag = (mask_frag.data > 0) & (~ann_mask_cutout_frag)
        sky_data_frag = ann_cutout_frag[valid_sky_mask_frag]
        
        sky_data_clipped_frag = sigma_clip(sky_data_frag, sigma=3, maxiters=10)
        msky_frag = np.ma.median(sky_data_clipped_frag)
        ssky_frag = np.ma.std(sky_data_clipped_frag)
        nsky_frag = sky_data_clipped_frag.count()

    # fits_summary.at[idx, "AP-SUM-OBJ"] = phot_obj['aperture_sum'][0] - msky * ap_obj.area
    ap_sum_corrected = phot_obj['aperture_sum'][0] - msky * eff_area_obj
    ap_sum_corrected_frag = phot_frag['aperture_sum'][0] - msky_frag * eff_area_frag
    # sqrt( Flux_err^2 + Area * stdev^2 + (Area^2 * stdev^2)/N_sky )
    if nsky > 0:
        ap_err_corrected = np.sqrt(
            phot_obj['aperture_sum_err'][0]**2 + 
            (eff_area_obj * ssky**2) + 
            ((eff_area_obj**2 * ssky**2) / nsky)
        )
    else:
        ap_err_corrected = np.nan
        
    if nsky_frag > 0:
        ap_err_corrected_frag = np.sqrt(
            phot_frag['aperture_sum_err'][0]**2 + 
            (eff_area_frag * ssky_frag**2) + 
            ((eff_area_frag**2 * ssky_frag**2) / nsky_frag)
        )
    else:
        ap_err_corrected_frag = np.nan
    # fits_summary.at[idx, "AP-ERR-OBJ"] = np.sqrt(phot_obj['aperture_sum_err'][0]**2 + ap_obj.area * ssky**2)
    fits_summary.at[idx, "AP-SUM-OBJ"] = ap_sum_corrected
    fits_summary.at[idx, "AP-ERR-OBJ"] = ap_err_corrected
    fits_summary.at[idx, "AP-NEFF"] = eff_area_obj
    fits_summary.at[idx, "AP-NREJ"] = ap_mask_cutout.sum() # number of masked pixels in the aperture
    
    fits_summary.at[idx, "AP-SUM-FRAG"] = ap_sum_corrected_frag
    fits_summary.at[idx, "AP-ERR-FRAG"] = ap_err_corrected_frag
    fits_summary.at[idx, "AP-NEFF-FRAG"] = eff_area_frag
    fits_summary.at[idx, "AP-NREJ-FRAG"] = ap_mask_cutout_frag.sum()
    
    #### Plotting
    ax = fig.add_subplot(gs[idx//5, idx%5])
    interval = ZScaleInterval()
    vmin, vmax = interval.get_limits(sci_masked)
    
    # cutout = Cutout2D(data, (row['XCEN'], row['YCEN']), (51, 51), wcs=wcs)
    
    ax.imshow(sci_masked, origin='lower', vmin=vmin, vmax=vmax, cmap='magma')
    ap_obj.plot(ax=ax, color='red', lw=2)
    an_obj.plot(ax=ax, color='red', lw=1, ls='--')
    ap_frag.plot(ax=ax, color='blue', lw=2)
    an_frag.plot(ax=ax, color='blue', lw=1, ls='--')
    
    ax.set_xlim(row['X-OBJ']-26, row['X-OBJ']+25)
    ax.set_ylim(row['Y-OBJ']-26, row['Y-OBJ']+25)
    ax.set_title(f"{row['R_HEL']:.2f} au, {row['WLEN-CEN']:.2f} um")
    ax.axis('off')
    
plt.show()

In [ ]:
fits_summary

In [ ]:
fits_summary.to_csv(DATA_DIR / f"{objname}_fits_summary.csv", index=False)

# Appendix

In [ ]:
phase = 1   
fits_summary_phased = fits_summary[fits_summary['PHASE'] == 1]
fits_summary_phased.sort_values('WLEN-CEN', inplace=True)

In [ ]:
fig  = plt.figure(figsize=(15, 6))
ax = fig.add_subplot()

x = fits_summary_phased["WLEN-CEN"]
apsum = fits_summary_phased["AP-SUM-OBJ"]
apsum_mjy = apsum * 1e9 * (fits_summary_phased["PIX-SCL"]/206265)**2
yerr = fits_summary_phased["AP-ERR-OBJ"]

ratio_neff = fits_summary_phased["AP-NEFF"] / (fits_summary_phased["AP-NEFF"] + fits_summary_phased["AP-NREJ"])
sc = ax.scatter(x, apsum_mjy, c=ratio_neff, cmap='jet', marker='o', zorder=5)

# 2. Plot the error bars separately (fmt='none' means no markers, just lines)
ax.errorbar(x, apsum_mjy, yerr=yerr, fmt='none', ecolor='gray', alpha=0.6, zorder=4)
ax.plot(x, apsum_mjy, lw=0.5, marker="none", alpha=0.7)

ax.axvspan(2.6, 2.8, color='skyblue', alpha=0.3) # H2O (2.7 um)
ax.axvspan(4.2, 4.4, color='skyblue', alpha=0.3) # CO2 (4.3 um)
ax.axvspan(4.6, 4.8, color='skyblue', alpha=0.3) # CO (4.7 um)

cbar = plt.colorbar(sc, ax=ax, pad=0.1, location='left')
cbar.set_label("$\mathrm{N_{neff} / (N_{neff} + N_{rej})}$")
ax.set_xlabel("Wavelength ($\\mu$m)")
ax.set_ylabel("ap_flux (mJy)")
date_obs_min = Time(fits_summary_phased['DATE-OBS'].min()).to_datetime().strftime('%Y-%m-%d')
date_obs_max = Time(fits_summary_phased['DATE-OBS'].max()).to_datetime().strftime('%Y-%m-%d')
ax.set_title(f"Phase {phase} ({date_obs_min} to {date_obs_max})")

ax.annotate(f"""{objname} ({len(fits_summary_phased)} data points)
$\\rho={fits_summary_phased['RHO_KM'].iloc[0]}~$ km
""",
            xy=(0.05, 0.95), xycoords='axes fraction',
            ha='left', va='top')

# obj_new = obj.replace(" ", "")

# plt.savefig(FIGDIR / f"{obj_new}_ap1sum.png")
# plt.close()
ax.set_ylim(top=50)
plt.savefig(FIG_DIR / f"{objname}_apsum_phase{phase}_neff.png")
plt.show()

In [ ]:
fig  = plt.figure(figsize=(15, 6))
ax = fig.add_subplot()

x = fits_summary_phased["WLEN-CEN"]
apsum = fits_summary_phased["AP-SUM-OBJ"]
apsum_mjy = apsum * 1e9 * (fits_summary_phased["PIX-SCL"]/206265)**2
yerr = fits_summary_phased["AP-ERR-OBJ"]

ratio_neff = fits_summary_phased["AP-NEFF"] / (fits_summary_phased["AP-NEFF"] + fits_summary_phased["AP-NREJ"])
# sc = ax.scatter(x, ap1_sum_mjy, c=ratio_neff, cmap='jet', marker='o', zorder=5)
sc = ax.scatter(x, apsum_mjy, c=fits_summary_phased['R-HEL'], cmap='RdBu', marker='o', zorder=5)

# 2. Plot the error bars separately (fmt='none' means no markers, just lines)
ax.errorbar(x, apsum_mjy, yerr=yerr, fmt='none', ecolor='gray', alpha=0.6, zorder=4)
ax.plot(x, apsum_mjy, lw=0.5, marker="none", alpha=0.7)

ax.axvspan(2.6, 2.8, color='skyblue', alpha=0.3) # H2O (2.7 um)
ax.axvspan(4.2, 4.4, color='skyblue', alpha=0.3) # CO2 (4.3 um)
ax.axvspan(4.6, 4.8, color='skyblue', alpha=0.3) # CO (4.7 um)

cbar = plt.colorbar(sc, ax=ax, pad=0.1, location='left')
# cbar.set_label("$\mathrm{N_{neff} / (N_{neff} + N_{rej})}$")
cbar.set_label("$R_{HEL}$ (au)")
ax.set_xlabel("Wavelength ($\\mu$m)")
ax.set_ylabel("ap_flux (mJy)")
date_obs_min = Time(fits_summary_phased['DATE-OBS'].min()).to_datetime().strftime('%Y-%m-%d')
date_obs_max = Time(fits_summary_phased['DATE-OBS'].max()).to_datetime().strftime('%Y-%m-%d')
ax.set_title(f"Phase {phase} ({date_obs_min} to {date_obs_max})")

ax.annotate(f"""{objname} ({len(fits_summary_phased)} data points)
$\\rho={fits_summary_phased['RHO_KM'].iloc[0]}~$ km
""",
            xy=(0.05, 0.95), xycoords='axes fraction',
            ha='left', va='top')

# obj_new = obj.replace(" ", "")

# plt.savefig(FIGDIR / f"{obj_new}_ap1sum.png")
# plt.close()
ax.set_ylim(top=50)
plt.savefig(FIG_DIR / f"{objname}_apsum_phase{phase}.png")
plt.show()

In [ ]:
fig  = plt.figure(figsize=(15, 6))
ax = fig.add_subplot()

x = fits_summary_phased["WLEN-CEN"]
apsum = fits_summary_phased["AP-SUM-OBJ"]
apsum_mjy = apsum * 1e9 * (fits_summary_phased["PIX-SCL"]/206265)**2
yerr = fits_summary_phased["AP-ERR-OBJ"]

apsum_frag = fits_summary_phased["AP-SUM-FRAG"]
apsum_frag_mjy = apsum_frag * 1e9 * (fits_summary_phased["PIX-SCL"]/206265)**2
yerr_frag = fits_summary_phased["AP-ERR-FRAG"]

ratio_neff = fits_summary_phased["AP-NEFF"] / (fits_summary_phased["AP-NEFF"] + fits_summary_phased["AP-NREJ"])
# sc = ax.scatter(x, apsum_mjy, c=ratio_neff, cmap='jet', marker='o', zorder=5)
sc = ax.scatter(x, apsum_mjy, color='k')

ratio_neff_frag = fits_summary_phased["AP-NEFF-FRAG"] / (fits_summary_phased["AP-NEFF-FRAG"] + fits_summary_phased["AP-NREJ-FRAG"])
# sc_frag = ax.scatter(x, apsum_frag_mjy, c=ratio_neff_frag, cmap='jet', marker='o', zorder=5)
sc_frag = ax.scatter(x, apsum_frag_mjy, color='r')

# sc = ax.scatter(x, ap1_sum_mjy, color='k')

# 2. Plot the error bars separately (fmt='none' means no markers, just lines)
ax.errorbar(x, apsum_mjy, yerr=yerr, fmt='none', ecolor='gray', alpha=0.6, zorder=4)
ax.plot(x, apsum_mjy, lw=1, marker="none", alpha=0.7, color='k', label='240P')

ax.errorbar(x, apsum_frag_mjy, yerr=yerr_frag, fmt='none', ecolor='pink', alpha=0.6, zorder=4)
ax.plot(x, apsum_frag_mjy, lw=1, marker="none", alpha=0.7, color='r', label='240P-B (fragment)')

ax.axvspan(2.6, 2.8, color='skyblue', alpha=0.3) # H2O (2.7 um)
ax.axvspan(4.2, 4.4, color='skyblue', alpha=0.3) # CO2 (4.3 um)
ax.axvspan(4.6, 4.8, color='skyblue', alpha=0.3) # CO (4.7 um)

# cbar = plt.colorbar(sc, ax=ax, pad=0.1, location='left')
# cbar.set_label("$\mathrm{N_{neff} / (N_{neff} + N_{rej})}$")
# cbar.set_label("$R_{HEL}$ (au)")
ax.set_xlabel("Wavelength ($\\mu$m)")
ax.set_ylabel("ap_flux (mJy)")
date_obs_min = Time(fits_summary_phased['DATE-OBS'].min()).to_datetime().strftime('%Y-%m-%d')
date_obs_max = Time(fits_summary_phased['DATE-OBS'].max()).to_datetime().strftime('%Y-%m-%d')
ax.set_title(f"Phase {phase} ({date_obs_min} to {date_obs_max})")

# ax.annotate(f"""{objname} ({len(fits_summary_phased)} data points)
# $\\rho={fits_summary_phased['RHO_KM'].iloc[0]}~$ km
# """,
#             xy=(0.05, 0.95), xycoords='axes fraction',
#             ha='left', va='top')

ax.legend()

# obj_new = obj.replace(" ", "")

# plt.savefig(FIGDIR / f"{obj_new}_ap1sum.png")
# plt.close()
ax.set_ylim(top=50)
plt.savefig(FIG_DIR / f"{objname}_apsum_phase{phase}_fragment.png")
plt.show()

In [ ]:
phase = 2
fits_summary_phased = fits_summary[fits_summary['PHASE'] == phase]
fits_summary_phased.sort_values('WLEN-CEN', inplace=True)

In [ ]:
fig  = plt.figure(figsize=(15, 6))
ax = fig.add_subplot()

x = fits_summary_phased["WLEN-CEN"]
apsum = fits_summary_phased["AP-SUM-OBJ"]
apsum_mjy = apsum * 1e9 * (fits_summary_phased["PIX-SCL"]/206265)**2
yerr = fits_summary_phased["AP-ERR-OBJ"]

ratio_neff = fits_summary_phased["AP-NEFF"] / (fits_summary_phased["AP-NEFF"] + fits_summary_phased["AP-NREJ"])
# sc = ax.scatter(x, ap1_sum_mjy, c=ratio_neff, cmap='jet', marker='o', zorder=5)
sc = ax.scatter(x, apsum_mjy, c=fits_summary_phased['R-HEL'], cmap='RdBu', marker='o', zorder=5)

# 2. Plot the error bars separately (fmt='none' means no markers, just lines)
ax.errorbar(x, apsum_mjy, yerr=yerr, fmt='none', ecolor='gray', alpha=0.6, zorder=4)
ax.plot(x, apsum_mjy, lw=0.5, marker="none", alpha=0.7)

ax.axvspan(2.6, 2.8, color='skyblue', alpha=0.3) # H2O (2.7 um)
ax.axvspan(4.2, 4.4, color='skyblue', alpha=0.3) # CO2 (4.3 um)
ax.axvspan(4.6, 4.8, color='skyblue', alpha=0.3) # CO (4.7 um)

cbar = plt.colorbar(sc, ax=ax, pad=0.1, location='left')
# cbar.set_label("$\mathrm{N_{neff} / (N_{neff} + N_{rej})}$")
cbar.set_label("$R_{HEL}$ (au)")
ax.set_xlabel("Wavelength ($\\mu$m)")
ax.set_ylabel("ap_flux (mJy)")
date_obs_min = Time(fits_summary_phased['DATE-OBS'].min()).to_datetime().strftime('%Y-%m-%d')
date_obs_max = Time(fits_summary_phased['DATE-OBS'].max()).to_datetime().strftime('%Y-%m-%d')
ax.set_title(f"Phase 2 ({date_obs_min} to {date_obs_max})")

ax.annotate(f"""{objname} ({len(fits_summary_phased)} data points)
$\\rho={fits_summary_phased['RHO_KM'].iloc[0]}~$ km
""",
            xy=(0.05, 0.95), xycoords='axes fraction',
            ha='left', va='top')

# obj_new = obj.replace(" ", "")

# plt.savefig(FIGDIR / f"{obj_new}_ap1sum.png")
# plt.close()
ax.set_ylim(top=50)
plt.savefig(FIG_DIR / f"{objname}_apsum_phase{phase}.png")
plt.show()

In [ ]:
fig  = plt.figure(figsize=(15, 6))
ax = fig.add_subplot()

x = fits_summary_phased["WLEN-CEN"]
apsum = fits_summary_phased["AP-SUM-OBJ"]
apsum_mjy = apsum * 1e9 * (fits_summary_phased["PIX-SCL"]/206265)**2
yerr = fits_summary_phased["AP-ERR-OBJ"]

apsum_frag = fits_summary_phased["AP-SUM-FRAG"]
apsum_frag_mjy = apsum_frag * 1e9 * (fits_summary_phased["PIX-SCL"]/206265)**2
yerr_frag = fits_summary_phased["AP-ERR-FRAG"]

ratio_neff = fits_summary_phased["AP-NEFF"] / (fits_summary_phased["AP-NEFF"] + fits_summary_phased["AP-NREJ"])
# sc = ax.scatter(x, apsum_mjy, c=ratio_neff, cmap='jet', marker='o', zorder=5)
sc = ax.scatter(x, apsum_mjy, color='k')

ratio_neff_frag = fits_summary_phased["AP-NEFF-FRAG"] / (fits_summary_phased["AP-NEFF-FRAG"] + fits_summary_phased["AP-NREJ-FRAG"])
# sc_frag = ax.scatter(x, apsum_frag_mjy, c=ratio_neff_frag, cmap='jet', marker='o', zorder=5)
sc_frag = ax.scatter(x, apsum_frag_mjy, color='r')

# sc = ax.scatter(x, ap1_sum_mjy, color='k')

# 2. Plot the error bars separately (fmt='none' means no markers, just lines)
ax.errorbar(x, apsum_mjy, yerr=yerr, fmt='none', ecolor='gray', alpha=0.6, zorder=4)
ax.plot(x, apsum_mjy, lw=1, marker="none", alpha=0.7, color='k', label='240P')

ax.errorbar(x, apsum_frag_mjy, yerr=yerr_frag, fmt='none', ecolor='pink', alpha=0.6, zorder=4)
ax.plot(x, apsum_frag_mjy, lw=1, marker="none", alpha=0.7, color='r', label='240P-B (fragment)')

ax.axvspan(2.6, 2.8, color='skyblue', alpha=0.3) # H2O (2.7 um)
ax.axvspan(4.2, 4.4, color='skyblue', alpha=0.3) # CO2 (4.3 um)
ax.axvspan(4.6, 4.8, color='skyblue', alpha=0.3) # CO (4.7 um)

# cbar = plt.colorbar(sc, ax=ax, pad=0.1, location='left')
# cbar.set_label("$\mathrm{N_{neff} / (N_{neff} + N_{rej})}$")
# cbar.set_label("$R_{HEL}$ (au)")
ax.set_xlabel("Wavelength ($\\mu$m)")
ax.set_ylabel("ap_flux (mJy)")
date_obs_min = Time(fits_summary_phased['DATE-OBS'].min()).to_datetime().strftime('%Y-%m-%d')
date_obs_max = Time(fits_summary_phased['DATE-OBS'].max()).to_datetime().strftime('%Y-%m-%d')
ax.set_title(f"Phase 2 ({date_obs_min} to {date_obs_max})")

# ax.annotate(f"""{objname} ({len(fits_summary_phased)} data points)
# $\\rho={fits_summary_phased['RHO_KM'].iloc[0]}~$ km
# """,
#             xy=(0.05, 0.95), xycoords='axes fraction',
#             ha='left', va='top')

ax.legend()

# obj_new = obj.replace(" ", "")

# plt.savefig(FIGDIR / f"{obj_new}_ap1sum.png")
# plt.close()
ax.set_ylim(top=30)
plt.savefig(FIG_DIR / f"{objname}_apsum_phase{phase}_fragment.png")
plt.show()

In [ ]:
# Gas emission lines (um)
dict_emission_um = {
    "H2O": (2.6, 2.8), # 2.7 um
    "CO2": (4.1, 4.4), # 4.25 (12CO2) and 4.4 (13CO2) um
    "CO" : (4.5, 4.8)  # 4.65 um
}

In [ ]:
molecules = "H2O"
wl_min, wl_max = dict_emission_um[molecules]
mask_wl = (fits_summary["WLEN-CEN"] >= wl_min) & (fits_summary["WLEN-CEN"] <= wl_max)
fits_summary_wl = fits_summary[mask_wl].reset_index(drop=True)
fits_summary_wl

### Images cutout example

In [ ]:
# cutout example
row = fits_summary_wl.loc[4]

hdul = fits.open(FITS_DIR / row['FILENAME'])
sci = hdul[1].data
flag = hdul[3].data # bitmap

cutout_size = (51, 51)  # pixels
cutout = Cutout2D(sci, (row['XCEN'], row['YCEN']), cutout_size, wcs=WCS(row['WCS_SERIALIZED']))
flag_cutout = Cutout2D(flag, (row['XCEN'], row['YCEN']), cutout_size, wcs=WCS(row['WCS_SERIALIZED']))

# xy center coordinates
# xycen_obj_cutout = cutout.wcs.world_to_pixel(SkyCoord(ra=row['RA-OBJ'], dec=row['DEC-OBJ'], unit='deg'))
# xycen_frag_cutout = cutout.wcs.world_to_pixel(SkyCoord(ra=row['RA-FRAG'], dec=row['DEC-FRAG'], unit='deg'))
xycen_obj_cutout = cutout.to_cutout_position((row['XCEN'], row['YCEN']))
xycen_frag_cutout = cutout.to_cutout_position((row['X-FRAG'], row['Y-FRAG']))

In [ ]:
# Contours example:
fig = plt.figure()
ax = fig.add_subplot(111)
interval = ZScaleInterval()
vmin, vmax = interval.get_limits(sci)
contour_levels = np.logspace(np.log10(vmin)+0.5, np.log10(vmax)+0.5, 5)
ax.imshow(sci, origin='lower', vmin=vmin, vmax=vmax, cmap='magma')
ax.contour(sci, levels=contour_levels, colors='green', origin='lower')
ax.scatter(row['X-OBJ'], row['Y-OBJ'], marker="x", s=100, color='red', label='240P/NEAT')
ax.scatter(row['X-FRAG'], row['Y-FRAG'], marker="x", s=100, color='blue', label='240P-B')

ax.set_title(f"{row['FILENAME']}")
ax.set_aspect('equal')

ax.set_xlim(row['XCEN']-26, row['XCEN']+25)
ax.set_ylim(row['YCEN']-26, row['YCEN']+25)

plt.show()

### Flagging

In [ ]:
flag_number = [0, 2, 6, 7, 9, 10, 11, 12, 14, 15, 17, 22, 24, 26, 27, 28, 29] # exclude: 19, 21
# 0: transient (e.g., cosmic ray)
# 1: overflow (half-saturated)
# 2: sur_error
# 6: permanently dead pixel
# 7: smile effect (band 3 and 4)
# 9: missing data
# 10: hot pixel
# 11: cold pixel
# 12: fullsample
# 14: phantom missing data
# 15: non-linear
# 17: affected by persistent charge
# 19: outlier pixel ==> likely transient
# 21: known source
# 22&24: affected by optical ghost
# 26: affected by source "blooming"
# 27: affected by "snowball" events
# 28: affected by "halo", especially around cosmic
# 29: affected by satellite streak
mask_obj = np.zeros_like(flag_cutout.data, dtype=bool)
for flag in flag_number:    
    mask_obj |= (flag_cutout.data & (1 << flag)) != 0
    mask_obj = mask_obj.astype(bool)
print(f"Total flagged pixels: {np.sum(mask_obj)}/{mask_obj.size}")
cutout_masked = np.ma.masked_array(cutout.data, mask=mask_obj)

In [ ]:
fig = plt.figure()
gs = GridSpec(1, 3, wspace=0.1)

ax1 = fig.add_subplot(gs[0])
ax1.imshow(cutout.data, origin='lower', vmin=vmin, vmax=vmax, cmap='magma')
ax1.set_title(f"Lv2 Cutout")

ax2 = fig.add_subplot(gs[1], sharex=ax1, sharey=ax1)
ax2.set_title(f"Bad pixels masked")
ax2.imshow(cutout_masked, origin='lower', vmin=vmin, vmax=vmax, cmap='magma')
ax2.tick_params(axis='both', which='both', labelbottom=False, labelleft=False)

# Contours
ax3 = fig.add_subplot(gs[2], sharex=ax1, sharey=ax1)
contour_levels = np.logspace(np.log10(vmin)+0.5, np.log10(vmax)+0.5, 5)
# ax.imshow(cutout_masked, origin='lower', vmin=vmin, vmax=vmax, cmap='magma')
ax3.contour(cutout_masked, levels=contour_levels, colors='green', origin='lower')
ax3.scatter(xycen_obj_cutout[0], xycen_obj_cutout[1], marker="x", s=100, color='red', label='240P/NEAT')
ax3.scatter(xycen_frag_cutout[0], xycen_frag_cutout[1], marker="x", s=100, color='blue', label='240P-B')
ax3.set_title(f"Contours")
ax3.set_aspect('equal')
ax3.tick_params(axis='both', which='both', labelbottom=False, labelleft=False)

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import binned_statistic

# 1. Get image dimensions and center coordinates
ny, nx = cutout_masked.shape
x_cen, y_cen = xycen_obj_cutout[0], xycen_obj_cutout[1]

# 2. Create a grid of x and y pixel coordinates
y, x = np.indices((ny, nx))

# 3. Calculate the radial distance of each pixel from the center
r = np.sqrt((x - x_cen)**2 + (y - y_cen)**2)

# 4. Flatten the arrays and handle the mask/NaNs
r_flat = r.flatten()
if np.ma.isMaskedArray(cutout_masked):
    flux_flat = cutout_masked.data.flatten()
    valid_pixels = ~cutout_masked.mask.flatten()
else:
    flux_flat = cutout_masked.flatten()
    valid_pixels = ~np.isnan(flux_flat)

r_valid = r_flat[valid_pixels]
flux_valid = flux_flat[valid_pixels]

# ---> NEW: Filter to keep only pixels within 10 pixels of the center <---
max_radius = 15
distance_mask = r_valid <= max_radius
r_zoomed = r_valid[distance_mask]
flux_zoomed = flux_valid[distance_mask]

# 5. Calculate a binned average up to 10 pixels
# bins = np.arange(0, max_radius + 1, 0.2)
bins = np.logspace(np.log10(0.1), np.log10(max_radius), 20)  # Logarithmic bins for better resolution at small radii
bin_centers = 0.5 * (bins[1:] + bins[:-1])

radial_profile, _, _ = binned_statistic(
    r_zoomed, flux_zoomed, statistic='mean', bins=bins
)

# 6. Plot the results
fig, ax = plt.subplots(figsize=(8, 6))

# Plot the raw scatter and binned average using the zoomed arrays
ax.scatter(r_zoomed, flux_zoomed, s=2, color='gray', alpha=0.3, label='Individual Pixels')
ax.plot(bin_centers, radial_profile, color='red', marker='o', lw=2, label='Binned Average')

ax.set_xlabel('Distance from Center (pixels)')
ax.set_ylabel('Surface Brightness')
ax.set_title('Radial Profile of 240P/NEAT (Inner 15 Pixels)')
ax.legend()
ax.grid(True, linestyle='--', alpha=0.6)
ax.set_yscale('log')  # Optional: use logarithmic scale for better visibility of faint features
# ax.set_xscale('log')
ax.set_ylim(bottom=1e-1)  # Adjust as needed based on your data range

# Force the x-axis to exactly match the 10-pixel limit
ax.set_xlim(0, max_radius)

plt.show()

In [ ]:
cutout_data = cutout.data.astype(np.float32)
bkg = sep.Background(cutout_data, mask=mask_obj)
source, segmap = sep.extract(cutout_data - bkg.back(),
                             thresh=1.5,
                             err=bkg.globalrms,
                             mask=mask_obj,
                             minarea=5,
                             deblend_cont=0.001,
                             segmentation_map=True
                             )
source

## Aperture photometry

In [ ]:
import astropy.units as u
from tqdm import tqdm

from photutils.aperture import CircularAperture, CircularAnnulus, aperture_photometry
from astropy.stats import sigma_clip

fits_summary["RHO_KM"] = 20000 # projected radius km

pixel_scale_km = (fits_summary["PIX-SCL"] * ((1*u.arcsec).to(u.rad).value) * fits_summary["R_OBS"] * (1*u.au).to(u.km).value) # km/pixel
fits_summary["RHO_PIX"] = fits_summary["RHO_KM"] / pixel_scale_km

# size of background annulus
fits_summary["AN_IN_PIX"]  = 6*fits_summary["RHO_PIX"]
fits_summary["AN_OUT_PIX"] = 8*fits_summary["RHO_PIX"]
# fits_summary["rho_fwhm"] = fits_summary["RHO_PIX"] / fits_summary["FWHM_PIX"]

In [ ]:
fig = plt.figure(figsize=(15, 3*(len(fits_summary)//5+1)))
gs  = GridSpec(nrows=len(fits_summary)//5+1, ncols=5, figure=fig)

for idx, row in tqdm(fits_summary.iterrows(), total=len(fits_summary), desc="Performing aperture photometry"):
    
    filepath = FITS_DIR / row['FILENAME']
    with fits.open(filepath) as hdul:
        sci = hdul[1].data.astype(np.float32)
        wcs  = WCS(row['WCS_SERIALIZED'])
        err  = np.sqrt(hdul[2].data.astype(np.float32)) # error array (variance^0.5)

    # skycoord = SkyCoord(ra=row.ra*u.deg, dec=row.dec*u.deg, frame='icrs')
    # xycoord = wcs.world_to_pixel(skycoord)

    # xycen_obj = np.array([row['XCEN'], row['YCEN']])
    # xycen_frag = np.array([row['XCEN-FRAG'], row['YCEN-FRAG']])

    # Refine object position with SEP.winpos
    # xycoord_winpos = sep.winpos(data, xinit=xycen_obj[0], yinit=xycen_obj[1], sig=3*row.fwhm_pix)

    # Define apertures
    ap_obj = CircularAperture((row['XCEN'], row['YCEN']), r=row['RHO_PIX'])
    an_obj  = CircularAnnulus((row['XCEN'], row['YCEN']), r_in=row['AN_IN_PIX'], r_out=row['AN_OUT_PIX'])

    ap_frag = CircularAperture((row['XCEN-FRAG'], row['YCEN-FRAG']), r=row['RHO_PIX'])
    an_frag = CircularAnnulus((row['XCEN-FRAG'], row['YCEN-FRAG']), r_in=row['AN_IN_PIX'], r_out=row['AN_OUT_PIX'])

    # Perform aperture photometry
    phot_obj  = aperture_photometry(sci, ap_obj, error=err)
    phot_frag = aperture_photometry(sci, ap_frag, error=err)

    # Estimate sky background from annulus
    mask_obj  = an_obj.to_mask(method='center')
    mask_frag = an_frag.to_mask(method='center')
    
    ann_obj   = mask_obj.multiply(sci)
    if ann_obj is None:
        msky = np.nan
        ssky = np.nan
        nsky = 0
    else:
        sky_data = ann_obj[mask_obj.data==1]
        sky_data_clipped = sigma_clip(sky_data, sigma=3, maxiters=10)
        med = np.ma.median(sky_data_clipped)
        stddev = np.ma.std(sky_data_clipped)
        msky = med
        ssky = stddev
        nsky = sky_data_clipped.count() # number of unmasked pixels

    fits_summary.at[idx, "AP-SUM-OBJ"] = phot_obj['aperture_sum'][0] - msky * ap_obj.area
    fits_summary.at[idx, "AP-ERR-OBJ"] = np.sqrt(phot_obj['aperture_sum_err'][0]**2 + ap_obj.area * ssky**2)
    
    ann_frag  = mask_frag.multiply(sci)
    if ann_frag is None:
        msky = np.nan
        ssky = np.nan
        nsky = 0
    else:
        sky_data = ann_frag[mask_frag.data==1]
        sky_data_clipped = sigma_clip(sky_data, sigma=3, maxiters=10)
        med = np.ma.median(sky_data_clipped)
        stddev = np.ma.std(sky_data_clipped)
        msky = med
        ssky = stddev
        nsky = sky_data_clipped.count()
        
    fits_summary.at[idx, "AP-SUM-FRAG"] = phot_frag['aperture_sum'][0] - msky * ap_frag.area
    fits_summary.at[idx, "AP-ERR-FRAG"] = np.sqrt(phot_frag['aperture_sum_err'][0]**2 + ap_frag.area * ssky**2)\
        
        
    #### 
    ax = fig.add_subplot(gs[idx//5, idx%5])
    interval = ZScaleInterval()
    vmin, vmax = interval.get_limits(sci)
    
    # cutout = Cutout2D(data, (row['XCEN'], row['YCEN']), (51, 51), wcs=wcs)
    
    ax.imshow(sci, origin='lower', vmin=vmin, vmax=vmax, cmap='gray')
    ap_obj.plot(ax=ax, color='red', lw=2)
    an_obj.plot(ax=ax, color='red', lw=1, ls='--')
    ap_frag.plot(ax=ax, color='blue', lw=2)
    an_frag.plot(ax=ax, color='blue', lw=1, ls='--')
    
    ax.set_xlim(row['XCEN']-26, row['XCEN']+25)
    ax.set_ylim(row['YCEN']-26, row['YCEN']+25)
    ax.set_title(f"{row['R_HEL']:.2f} au, {row['WLEN-CEN']:.2f} um")
    ax.axis('off')
    
plt.show()

In [ ]:
fits_summary

In [ ]:
fits_summary_01 = fits_summary[fits_summary['PHASE'] == 1].reset_index(drop=True)
fits_summary_01.sort_values('WLEN-CEN', inplace=True)
fits_summary_01

In [ ]:
fits_summary_01['OBSWEEKID'].value_counts()

In [ ]:
fits_summary_01['R_HEL'].describe()

In [ ]:
fig  = plt.figure(figsize=(15, 6))
ax = fig.add_subplot()

x = fits_summary_01["WLEN-CEN"]
apsum = fits_summary_01["AP-SUM-OBJ"]
apsum_mjy = apsum * 1e9 * (fits_summary_01["PIX-SCL"]/206265)**2
yerr = fits_summary_01["AP-ERR-OBJ"]

# pix_scale_km = df_target["pix_scale"]*u.arcsec.to(u.rad)* df_target["r_obs"]*u.au.to(u.km)
# ap1_rad_km = df_target["ap1_rad"] * pix_scale_km

# obs_ids = df_target["obsid"].apply(lambda x: x.split("_")[0]).unique()
# obs_ids_min = obs_ids.min()
# obs_ids_max = obs_ids.max()
# 1. Plot the colored points using scatter (returns the mappable for the colorbar)
sc = ax.scatter(x, apsum_mjy, c=fits_summary_01["R_HEL"], cmap='RdBu', marker='o', zorder=5)

# 2. Plot the error bars separately (fmt='none' means no markers, just lines)
ax.errorbar(x, apsum_mjy, yerr=yerr, fmt='none', ecolor='gray', alpha=0.6, zorder=4)

ax.axvspan(2.6, 2.8, color='skyblue', alpha=0.3) # H2O (2.7 um)
ax.axvspan(4.2, 4.4, color='skyblue', alpha=0.3) # CO2 (4.3 um)
ax.axvspan(4.6, 4.8, color='skyblue', alpha=0.3) # CO (4.7 um)

ax.plot(x, apsum_mjy, lw=0.5, marker="none", alpha=0.7, label=f"Comet")
cbar = plt.colorbar(sc, ax=ax, pad=0.1, location='left')
cbar.set_label("r_hel (au)")
ax.set_xlabel("Wavelength ($\\mu$m)")
ax.set_ylabel("ap1_sum (mJy)")
# # ax.set_title(f"Target Object Spectrum: {obj}")

# ax.annotate(f"""Target={obj} ({len(df_target)} data points)
# {obs_ids_min} to {obs_ids_max}

# $r_{{ap1}}=${ap1_rad_km.min():.1f} - {ap1_rad_km.max():.1f} km (2 pix)
# $V_{{mag}}=${df_target["vmag"].min():.1f} - {df_target["vmag"].max():.1f} mag
# $\\Delta r_{{h}}=${df_target["r_hel"].max() - df_target["r_hel"].min():.2f} au
# """,
#             xy=(1.05, 0.95), xycoords='axes fraction',
#             ha='left', va='top')

# obj_new = obj.replace(" ", "")

# plt.savefig(FIGDIR / f"{obj_new}_ap1sum.png")
# plt.close()
plt.show()

In [ ]:
fig  = plt.figure(figsize=(15, 6))
ax = fig.add_subplot()

x = fits_summary_01["WLEN-CEN"]
apsum = fits_summary_01["AP-SUM-FRAG"]
apsum_mjy = apsum * 1e9 * (fits_summary_01["PIX-SCL"]/206265)**2
yerr = fits_summary_01["AP-ERR-FRAG"]

# pix_scale_km = df_target["pix_scale"]*u.arcsec.to(u.rad)* df_target["r_obs"]*u.au.to(u.km)
# ap1_rad_km = df_target["ap1_rad"] * pix_scale_km

# obs_ids = df_target["obsid"].apply(lambda x: x.split("_")[0]).unique()
# obs_ids_min = obs_ids.min()
# obs_ids_max = obs_ids.max()
# 1. Plot the colored points using scatter (returns the mappable for the colorbar)
sc = ax.scatter(x, apsum_mjy, c=fits_summary_01["R_HEL"], cmap='RdBu', marker='o', zorder=5)

# 2. Plot the error bars separately (fmt='none' means no markers, just lines)
ax.errorbar(x, apsum_mjy, yerr=yerr, fmt='none', ecolor='gray', alpha=0.6, zorder=4)

ax.axvspan(2.6, 2.8, color='skyblue', alpha=0.3) # H2O (2.7 um)
ax.axvspan(4.2, 4.4, color='skyblue', alpha=0.3) # CO2 (4.3 um)
ax.axvspan(4.6, 4.8, color='skyblue', alpha=0.3) # CO (4.7 um)

ax.plot(x, apsum_mjy, lw=0.5, marker="none", alpha=0.7, label=f"Comet")
cbar = plt.colorbar(sc, ax=ax, pad=0.1, location='left')
cbar.set_label("r_hel (au)")
ax.set_xlabel("Wavelength ($\\mu$m)")
ax.set_ylabel("ap1_sum (mJy)")
# # ax.set_title(f"Target Object Spectrum: {obj}")

# ax.annotate(f"""Target={obj} ({len(df_target)} data points)
# {obs_ids_min} to {obs_ids_max}

# $r_{{ap1}}=${ap1_rad_km.min():.1f} - {ap1_rad_km.max():.1f} km (2 pix)
# $V_{{mag}}=${df_target["vmag"].min():.1f} - {df_target["vmag"].max():.1f} mag
# $\\Delta r_{{h}}=${df_target["r_hel"].max() - df_target["r_hel"].min():.2f} au
# """,
#             xy=(1.05, 0.95), xycoords='axes fraction',
#             ha='left', va='top')

# obj_new = obj.replace(" ", "")

# plt.savefig(FIGDIR / f"{obj_new}_ap1sum.png")
# plt.close()
ax.set_ylim(-4, 10)
plt.show()

In [ ]:
fits_summary_02 = fits_summary[fits_summary['PHASE'] == 2].reset_index(drop=True)
fits_summary_02.sort_values('WLEN-CEN', inplace=True)
fits_summary_02

In [ ]:
fits_summary_02['OBSWEEKID'].value_counts()

In [ ]:
fits_summary_02['R_HEL'].describe()

In [ ]:
fig  = plt.figure(figsize=(15, 6))
ax = fig.add_subplot()

x = fits_summary_02["WLEN-CEN"]
apsum = fits_summary_02["AP-SUM-OBJ"]
apsum_mjy = apsum * 1e9 * (fits_summary_02["PIX-SCL"]/206265)**2
yerr = fits_summary_02["AP-ERR-OBJ"]

# pix_scale_km = df_target["pix_scale"]*u.arcsec.to(u.rad)* df_target["r_obs"]*u.au.to(u.km)
# ap1_rad_km = df_target["ap1_rad"] * pix_scale_km

# Masking abnormally high values (e.g., due to contamination or artifacts) to focus on the main distribution
mask_data = apsum_mjy < 50
x = x[mask_data]
apsum_mjy = apsum_mjy[mask_data]
yerr = yerr[mask_data]

# obs_ids = df_target["obsid"].apply(lambda x: x.split("_")[0]).unique()
# obs_ids_min = obs_ids.min()
# obs_ids_max = obs_ids.max()
# 1. Plot the colored points using scatter (returns the mappable for the colorbar)
sc = ax.scatter(x, apsum_mjy, c=fits_summary_02["R_HEL"][mask_data], cmap='RdBu', marker='o', zorder=5)

# 2. Plot the error bars separately (fmt='none' means no markers, just lines)
ax.errorbar(x, apsum_mjy, yerr=yerr, fmt='none', ecolor='gray', alpha=0.6, zorder=4)

ax.axvspan(2.6, 2.8, color='skyblue', alpha=0.3) # H2O (2.7 um)
ax.axvspan(4.2, 4.4, color='skyblue', alpha=0.3) # CO2 (4.3 um)
ax.axvspan(4.6, 4.8, color='skyblue', alpha=0.3) # CO (4.7 um)

ax.plot(x, apsum_mjy, lw=0.5, marker="none", alpha=0.7, label=f"Comet")
cbar = plt.colorbar(sc, ax=ax, pad=0.1, location='left')
cbar.set_label("r_hel (au)")
ax.set_xlabel("Wavelength ($\\mu$m)")
ax.set_ylabel("ap1_sum (mJy)")
# # ax.set_title(f"Target Object Spectrum: {obj}")

# ax.annotate(f"""Target={obj} ({len(df_target)} data points)
# {obs_ids_min} to {obs_ids_max}

# $r_{{ap1}}=${ap1_rad_km.min():.1f} - {ap1_rad_km.max():.1f} km (2 pix)
# $V_{{mag}}=${df_target["vmag"].min():.1f} - {df_target["vmag"].max():.1f} mag
# $\\Delta r_{{h}}=${df_target["r_hel"].max() - df_target["r_hel"].min():.2f} au
# """,
#             xy=(1.05, 0.95), xycoords='axes fraction',
#             ha='left', va='top')

# obj_new = obj.replace(" ", "")

# plt.savefig(FIGDIR / f"{obj_new}_ap1sum.png")
# plt.close()
ax.set_ylim(top=50)
plt.show()

In [ ]:
fig  = plt.figure(figsize=(15, 6))
ax = fig.add_subplot()

x = fits_summary_02["WLEN-CEN"]
apsum = fits_summary_02["AP-SUM-FRAG"]
apsum_mjy = apsum * 1e9 * (fits_summary_02["PIX-SCL"]/206265)**2
yerr = fits_summary_02["AP-ERR-FRAG"]

# Masking abnormally high values (e.g., due to contamination or artifacts) to focus on the main distribution
mask_data = apsum_mjy < 20
x = x[mask_data]
apsum_mjy = apsum_mjy[mask_data]
yerr = yerr[mask_data]

# pix_scale_km = df_target["pix_scale"]*u.arcsec.to(u.rad)* df_target["r_obs"]*u.au.to(u.km)
# ap1_rad_km = df_target["ap1_rad"] * pix_scale_km

# obs_ids = df_target["obsid"].apply(lambda x: x.split("_")[0]).unique()
# obs_ids_min = obs_ids.min()
# obs_ids_max = obs_ids.max()
# 1. Plot the colored points using scatter (returns the mappable for the colorbar)
sc = ax.scatter(x, apsum_mjy, c=fits_summary_02["R_HEL"][mask_data], cmap='RdBu', marker='o', zorder=5)

# 2. Plot the error bars separately (fmt='none' means no markers, just lines)
ax.errorbar(x, apsum_mjy, yerr=yerr, fmt='none', ecolor='gray', alpha=0.6, zorder=4)

ax.axvspan(2.6, 2.8, color='skyblue', alpha=0.3) # H2O (2.7 um)
ax.axvspan(4.2, 4.4, color='skyblue', alpha=0.3) # CO2 (4.3 um)
ax.axvspan(4.6, 4.8, color='skyblue', alpha=0.3) # CO (4.7 um)

ax.plot(x, apsum_mjy, lw=0.5, marker="none", alpha=0.7, label=f"Comet")
cbar = plt.colorbar(sc, ax=ax, pad=0.1, location='left')
cbar.set_label("r_hel (au)")
ax.set_xlabel("Wavelength ($\\mu$m)")
ax.set_ylabel("ap1_sum (mJy)")
# # ax.set_title(f"Target Object Spectrum: {obj}")

# ax.annotate(f"""Target={obj} ({len(df_target)} data points)
# {obs_ids_min} to {obs_ids_max}

# $r_{{ap1}}=${ap1_rad_km.min():.1f} - {ap1_rad_km.max():.1f} km (2 pix)
# $V_{{mag}}=${df_target["vmag"].min():.1f} - {df_target["vmag"].max():.1f} mag
# $\\Delta r_{{h}}=${df_target["r_hel"].max() - df_target["r_hel"].min():.2f} au
# """,
#             xy=(1.05, 0.95), xycoords='axes fraction',
#             ha='left', va='top')

# obj_new = obj.replace(" ", "")

# plt.savefig(FIGDIR / f"{obj_new}_ap1sum.png")
# plt.close()
ax.set_ylim(-4, 20)
plt.show()

In [ ]:
import numpy as np
from scipy.optimize import curve_fit

# Define a Gaussian function for the emission line
def gaussian(x, amp, cen, wid):
    return amp * np.exp(-(x - cen)**2 / (2 * wid**2))

def fit_and_plot_feature(ax, x_data, y_data, wave_min, wave_max, color, label_prefix, cont_width=0.15):
    """
    Fits a local linear continuum and a Gaussian emission line, then plots them.
    """
    # 1. Define masks for the line and the local continuum (shoulders)
    line_mask = (x_data >= wave_min) & (x_data <= wave_max)
    cont_mask = ((x_data >= wave_min - cont_width) & (x_data < wave_min)) | \
                ((x_data > wave_max) & (x_data <= wave_max + cont_width))
    
    if not np.any(line_mask) or not np.any(cont_mask):
        return # Skip if data is missing in this range
        
    # 2. Fit the local continuum
    x_cont = x_data[cont_mask]
    y_cont = y_data[cont_mask]
    
    if len(x_cont) < 2:
        return # Need at least 2 points for a linear fit
        
    cont_coeffs = np.polyfit(x_cont, y_cont, 1)
    cont_poly = np.poly1d(cont_coeffs)
    
    # 3. Subtract continuum to isolate the line flux
    x_line = x_data[line_mask]
    y_line_raw = y_data[line_mask]
    y_line_sub = y_line_raw - cont_poly(x_line)
    
    # 4. Fit Gaussian to the continuum-subtracted line
    # Initial guesses: max height, center of the window, and a small width
    guess_amp = np.max(y_line_sub) if len(y_line_sub) > 0 else 1.0
    guess_cen = np.mean([wave_min, wave_max])
    guess_wid = 0.05
    
    try:
        popt, _ = curve_fit(gaussian, x_line, y_line_sub, p0=[guess_amp, guess_cen, guess_wid])
        
        # 5. Plot the Continuum (Dotted line)
        x_plot_cont = np.linspace(wave_min - cont_width, wave_max + cont_width, 100)
        ax.plot(x_plot_cont, cont_poly(x_plot_cont), color=color, linestyle=':', alpha=0.7)
        
        # 6. Plot the Full Model: Continuum + Gaussian (Thick solid line)
        x_plot_line = np.linspace(wave_min, wave_max, 100)
        y_plot_full = cont_poly(x_plot_line) + gaussian(x_plot_line, *popt)
        ax.plot(x_plot_line, y_plot_full, color=color, linewidth=2.5, zorder=10)
        
    except RuntimeError:
        print(f"Optimal parameters not found for {label_prefix} between {wave_min}-{wave_max} um.")


In [ ]:
fig = plt.figure(figsize=(15, 6))

ax = fig.add_subplot()

x = fits_summary_01["WLEN-CEN"]

apsum = fits_summary_01["AP-SUM-OBJ"]
apsum_frag = fits_summary_01["AP-SUM-FRAG"]

apsum_mjy = apsum * 1e9 * (fits_summary_01["PIX-SCL"]/206265)**2
ap1_sum_frag_mjy = apsum_frag * 1e9 * (fits_summary_01["PIX-SCL"]/206265)**2

yerr = fits_summary_01["AP-ERR-OBJ"]
yerr_frag = fits_summary_01["AP-ERR-FRAG"]

# Masking abnormally high values (e.g., due to contamination or artifacts) to focus on the main distribution
mask_data = ~((apsum_mjy > 50) | (ap1_sum_frag_mjy > 10))

x = x[mask_data]
ap1_sum_frag_mjy = ap1_sum_frag_mjy[mask_data]
apsum_mjy = apsum_mjy[mask_data]
yerr = yerr[mask_data]
yerr_frag = yerr_frag[mask_data]

# pix_scale_km = df_target["pix_scale"]*u.arcsec.to(u.rad)* df_target["r_obs"]*u.au.to(u.km)
# ap1_rad_km = df_target["ap1_rad"] * pix_scale_km

# obs_ids = df_target["obsid"].apply(lambda x: x.split("_")[0]).unique()
# obs_ids_min = obs_ids.min()
# obs_ids_max = obs_ids.max()
# 1. Plot the colored points using scatter (returns the mappable for the colorbar)

# sc = ax.scatter(x, ap1_sum_mjy, c=fits_summary_01["R_HEL"][mask_data], cmap='RdBu', marker='o', zorder=5)
# sc_frag = ax.scatter(x, ap1_sum_frag_mjy, c=fits_summary_01["R_HEL"][mask_data], cmap='RdBu', marker='x', zorder=5)

sc = ax.scatter(x, apsum_mjy, zorder=5, color='k')
sc_frag = ax.scatter(x, ap1_sum_frag_mjy, zorder=5, color='red')

# 2. Plot the error bars separately (fmt='none' means no markers, just lines)
ax.errorbar(x, apsum_mjy, yerr=yerr, fmt='none', ecolor='k', alpha=0.6, zorder=4)
ax.errorbar(x, ap1_sum_frag_mjy, yerr=yerr_frag, fmt='none', ecolor='red', alpha=0.6, zorder=4)

ax.axvspan(2.6, 2.8, color='skyblue', alpha=0.3) # H2O (2.7 um)
ax.axvspan(4.2, 4.4, color='skyblue', alpha=0.3) # CO2 (4.3 um)
ax.axvspan(4.6, 4.8, color='skyblue', alpha=0.3) # CO (4.7 um)

ax.plot(x, apsum_mjy, lw=1.5, color='k', marker="none", alpha=0.7, label=f"240P")
ax.plot(x, ap1_sum_frag_mjy, lw=1.5, color='red', marker="none", alpha=0.7, label=f"240P-B (Fragment)", ls='--')
# cbar = plt.colorbar(sc, ax=ax, pad=0.1, location='left')
# cbar.set_label("r_hel (au)")
ax.set_xlabel("Wavelength ($\\mu$m)")
ax.set_ylabel("Aperture Flux (mJy)")
ax.legend()

# --- Execute Fitting for H2O and CO2 ---

# H2O feature (~2.6 - 2.8 um)
fit_and_plot_feature(ax, x, apsum_mjy, 2.57, 2.88, 'k', '240P H2O')
fit_and_plot_feature(ax, x, ap1_sum_frag_mjy, 2.57, 2.88, 'red', '240P-B H2O')

# CO2 feature (~4.2 - 4.4 um)
fit_and_plot_feature(ax, x, apsum_mjy, 4.2, 4.4, 'k', '240P CO2')
fit_and_plot_feature(ax, x, ap1_sum_frag_mjy, 4.2, 4.4, 'red', '240P-B CO2')

# ax.set_yscale('log')
# # ax.set_title(f"Target Object Spectrum: {obj}")

# ax.annotate(f"""Target={obj} ({len(df_target)} data points)
# {obs_ids_min} to {obs_ids_max}

# $r_{{ap1}}=${ap1_rad_km.min():.1f} - {ap1_rad_km.max():.1f} km (2 pix)
# $V_{{mag}}=${df_target["vmag"].min():.1f} - {df_target["vmag"].max():.1f} mag
# $\\Delta r_{{h}}=${df_target["r_hel"].max() - df_target["r_hel"].min():.2f} au
# """,
#             xy=(1.05, 0.95), xycoords='axes fraction',
#             ha='left', va='top')

# obj_new = obj.replace(" ", "")

# plt.savefig(FIGDIR / f"{obj_new}_ap1sum.png")
# plt.close()
ax.set_ylim(-4, 10)
# plt.savefig(FIG_DIR / f"240P_240P-B_ap1sum_comparison_01_fitting.png")
plt.show()

In [ ]:
fig = plt.figure(figsize=(10, 6))

ax = fig.add_subplot()

x = fits_summary_02["WLEN-CEN"]

apsum = fits_summary_02["AP-SUM-OBJ"]
apsum_frag = fits_summary_02["AP-SUM-FRAG"]

apsum_mjy = apsum * 1e9 * (fits_summary_02["PIX-SCL"]/206265)**2
ap1_sum_frag_mjy = apsum_frag * 1e9 * (fits_summary_02["PIX-SCL"]/206265)**2

yerr = fits_summary_02["AP-ERR-OBJ"]
yerr_frag = fits_summary_02["AP-ERR-FRAG"]

# Masking abnormally high values (e.g., due to contamination or artifacts) to focus on the main distribution
mask_data = ~((apsum_mjy > 50) | (ap1_sum_frag_mjy > 10) | (apsum_mjy.isna()) | (ap1_sum_frag_mjy.isna()))

x = x[mask_data]
ap1_sum_frag_mjy = ap1_sum_frag_mjy[mask_data]
apsum_mjy = apsum_mjy[mask_data]
yerr = yerr[mask_data]
yerr_frag = yerr_frag[mask_data]

# pix_scale_km = df_target["pix_scale"]*u.arcsec.to(u.rad)* df_target["r_obs"]*u.au.to(u.km)
# ap1_rad_km = df_target["ap1_rad"] * pix_scale_km

# obs_ids = df_target["obsid"].apply(lambda x: x.split("_")[0]).unique()
# obs_ids_min = obs_ids.min()
# obs_ids_max = obs_ids.max()
# 1. Plot the colored points using scatter (returns the mappable for the colorbar)

# sc = ax.scatter(x, ap1_sum_mjy, c=fits_summary_02["R_HEL"][mask_data], cmap='RdBu', marker='o', zorder=5)
# sc_frag = ax.scatter(x, ap1_sum_frag_mjy, c=fits_summary_02["R_HEL"][mask_data], cmap='RdBu', marker='x', zorder=5)

sc = ax.scatter(x, apsum_mjy, zorder=5, color='k')
sc_frag = ax.scatter(x, ap1_sum_frag_mjy, zorder=5, color='red')

# 2. Plot the error bars separately (fmt='none' means no markers, just lines)
ax.errorbar(x, apsum_mjy, yerr=yerr, fmt='none', ecolor='k', alpha=0.6, zorder=4)
ax.errorbar(x, ap1_sum_frag_mjy, yerr=yerr_frag, fmt='none', ecolor='red', alpha=0.6, zorder=4)

ax.axvspan(2.6, 2.8, color='skyblue', alpha=0.3) # H2O (2.7 um)
ax.axvspan(4.2, 4.4, color='skyblue', alpha=0.3) # CO2 (4.3 um)
ax.axvspan(4.6, 4.8, color='skyblue', alpha=0.3) # CO (4.7 um)

ax.plot(x, apsum_mjy, lw=1.5, color='k', marker="none", alpha=0.7, label=f"240P")
ax.plot(x, ap1_sum_frag_mjy, lw=1.5, color='red', marker="none", alpha=0.7, label=f"240P-B (Fragment)", ls='--')
# cbar = plt.colorbar(sc, ax=ax, pad=0.1, location='left')
# cbar.set_label("r_hel (au)")
ax.set_xlabel("Wavelength ($\\mu$m)")
ax.set_ylabel("Aperture Flux (mJy)")
ax.legend()
# ax.set_yscale('log')
# # ax.set_title(f"Target Object Spectrum: {obj}")

# ax.annotate(f"""Target={obj} ({len(df_target)} data points)
# {obs_ids_min} to {obs_ids_max}

# $r_{{ap1}}=${ap1_rad_km.min():.1f} - {ap1_rad_km.max():.1f} km (2 pix)
# $V_{{mag}}=${df_target["vmag"].min():.1f} - {df_target["vmag"].max():.1f} mag
# $\\Delta r_{{h}}=${df_target["r_hel"].max() - df_target["r_hel"].min():.2f} au
# """,
#             xy=(1.05, 0.95), xycoords='axes fraction',
#             ha='left', va='top')

ax.set_xlim(4.0, 4.7)

# # obj_new = obj.replace(" ", "")
# # H2O feature (~2.6 - 2.8 um)
fit_and_plot_feature(ax, x, apsum_mjy, 2.52, 2.93, 'k', '240P H2O')
fit_and_plot_feature(ax, x, ap1_sum_frag_mjy, 2.52, 2.93, 'red', '240P-B H2O')

# CO2 feature (~4.2 - 4.4 um)
fit_and_plot_feature(ax, x, apsum_mjy, 4.1, 4.35, 'k', '240P CO2')
fit_and_plot_feature(ax, x, ap1_sum_frag_mjy, 4.1, 4.4, 'red', '240P-B CO2')


# plt.savefig(FIGDIR / f"{obj_new}_ap1sum.png")
# plt.close()
ax.set_ylim(-4, 30)
plt.savefig(FIG_DIR / f"240P_240P-B_ap1sum_comparison_02_cutout.png")
plt.show()

In [ ]:
import numpy as np
from scipy.optimize import curve_fit

def gaussian(x, amp, cen, wid):
    return amp * np.exp(-(x - cen)**2 / (2 * wid**2))

def fit_and_integrate_feature(ax, x_data, y_data, wave_min, wave_max, color, label_prefix, cont_width=0.15):
    """
    Fits a local continuum and Gaussian line, plots them, and calculates the integrated flux.
    """
    # --- THE FIX: Convert to pure NumPy arrays to avoid pandas indexing errors ---
    x_data = np.asarray(x_data)
    y_data = np.asarray(y_data)
    
    # 1. Define masks
    line_mask = (x_data >= wave_min) & (x_data <= wave_max)
    cont_mask = ((x_data >= wave_min - cont_width) & (x_data < wave_min)) | \
                ((x_data > wave_max) & (x_data <= wave_max + cont_width))
    
    if not np.any(line_mask) or not np.any(cont_mask):
        return None, None
        
    # 2. Fit continuum
    x_cont = x_data[cont_mask]
    y_cont = y_data[cont_mask]
    
    if len(x_cont) < 2:
        return None, None
        
    cont_coeffs = np.polyfit(x_cont, y_cont, 1)
    cont_poly = np.poly1d(cont_coeffs)
    
    # 3. Continuum subtraction
    x_line = x_data[line_mask]
    y_line_raw = y_data[line_mask]
    y_line_sub = y_line_raw - cont_poly(x_line)
    
    # --- NUMERICAL INTEGRATION ---
    # Now that they are pure NumPy arrays, integer indexing works perfectly.
    sort_idx = np.argsort(x_line)
    area_numerical = np.trapz(y_line_sub[sort_idx], x_line[sort_idx])
    
    # 4. Fit Gaussian
    guess_amp = np.max(y_line_sub) if len(y_line_sub) > 0 else 1.0
    guess_cen = np.mean([wave_min, wave_max])
    guess_wid = 0.05
    
    try:
        popt, _ = curve_fit(gaussian, x_line, y_line_sub, p0=[guess_amp, guess_cen, guess_wid])
        
        # --- ANALYTICAL INTEGRATION ---
        amp, cen, wid = popt
        area_analytical = amp * np.abs(wid) * np.sqrt(2 * np.pi)
        
        print(f"--- {label_prefix} ---")
        print(f"Numerical Area (Data):  {area_numerical:.4f} mJy * um")
        print(f"Analytical Area (Fit):  {area_analytical:.4f} mJy * um")
        print(f"Fit Parameters: Amp={amp:.2f}, Cen={cen:.3f}, Width={wid:.3f}\n")
        
        # 5. Plotting
        x_plot_cont = np.linspace(wave_min - cont_width, wave_max + cont_width, 100)
        ax.plot(x_plot_cont, cont_poly(x_plot_cont), color=color, linestyle=':', alpha=0.7)
        
        x_plot_line = np.linspace(wave_min, wave_max, 100)
        y_plot_full = cont_poly(x_plot_line) + gaussian(x_plot_line, *popt)
        ax.plot(x_plot_line, y_plot_full, color=color, linewidth=2.5, zorder=10)
        
        return area_numerical, area_analytical
        
    except RuntimeError:
        print(f"Failed to fit {label_prefix} between {wave_min}-{wave_max} um.")
        return area_numerical, None

In [ ]:
fit_and_integrate_feature(ax, x, apsum_mjy, 4.1, 4.35, 'k', '240P CO2')
fit_and_integrate_feature(ax, x, ap1_sum_frag_mjy, 4.2, 4.4, 'red', '240P-B CO2')

In [ ]:
# Gas emission lines (um)
dict_emission_um = {
    "Continuum" : (1.5, 2.2), # continuum 
    "H2O": (2.6, 2.8), # 2.7 um
    "CO2": (4.2, 4.4), # 4.25 (12CO2) and 4.4 (13CO2) um
    "CO" : (4.5, 4.8)  # 4.65 um
}

In [ ]:
molecules = "Continuum"
wl_min, wl_max = dict_emission_um[molecules]
mask_wl = (fits_summary_01["WLEN-CEN"] >= wl_min) & (fits_summary_01["WLEN-CEN"] <= wl_max)
fits_summary_01_wl = fits_summary_01[mask_wl].reset_index(drop=True)
fits_summary_01_wl

In [ ]:
# cutout example
cutout_data_combined = []
for idx, row in fits_summary_01_wl.iterrows():
    
    row = fits_summary_01_wl.loc[0]

    hdul = fits.open(FITS_DIR / row['FILENAME'])
    sci = hdul[1].data
    flag = hdul[3].data # bitmap

    cutout_size = (51, 51)  # pixels
    cutout = Cutout2D(sci, (row['XCEN'], row['YCEN']), cutout_size, wcs=WCS(row['WCS_SERIALIZED']))
    flag_cutout = Cutout2D(flag, (row['XCEN'], row['YCEN']), cutout_size, wcs=WCS(row['WCS_SERIALIZED']))

    # xy center coordinates
    # xycen_obj_cutout = cutout.wcs.world_to_pixel(SkyCoord(ra=row['RA-OBJ'], dec=row['DEC-OBJ'], unit='deg'))
    # xycen_frag_cutout = cutout.wcs.world_to_pixel(SkyCoord(ra=row['RA-FRAG'], dec=row['DEC-FRAG'], unit='deg'))
    xycen_obj_cutout = cutout.to_cutout_position((row['XCEN'], row['YCEN']))
    xycen_frag_cutout = cutout.to_cutout_position((row['XCEN-FRAG'], row['YCEN-FRAG']))

    cutout_data_combined.append(cutout.data)

cutout_data_combined = np.array(cutout_data_combined)
cutout_data_combined = cutout_data_combined.mean(axis=0) # average cutout

contour_levels = np.percentile(cutout_data_combined, [50, 70, 90])

fig, ax = plt.subplots(figsize=(6, 6))
interval = ZScaleInterval()
vmin, vmax = interval.get_limits(cutout_data_combined)
ax.imshow(cutout_data_combined, origin='lower', vmin=vmin, vmax=vmax, cmap='magma')
ax.contour(cutout_data_combined, levels=contour_levels, colors='white', linewidths=0.5)
ax.scatter(xycen_obj_cutout[0], xycen_obj_cutout[1], color='red', marker='X', s=200, label='Object Center')
ax.scatter(xycen_frag_cutout[0], xycen_frag_cutout[1], color='blue', marker='X', s=200, label='Fragment Center')
# ax.legend()
ax.set_title(f"{molecules} ({wl_min}-{wl_max} um, n={len(fits_summary_01_wl)})")
ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
plt.savefig(FIG_DIR / f"cutout_combined_01_{molecules}.png")
plt.show()

In [ ]:
molecules = "H2O"
wl_min, wl_max = dict_emission_um[molecules]
mask_wl = (fits_summary_01["WLEN-CEN"] >= wl_min) & (fits_summary_01["WLEN-CEN"] <= wl_max)
fits_summary_01_wl = fits_summary_01[mask_wl].reset_index(drop=True)
fits_summary_01_wl

In [ ]:
# cutout example
cutout_data_combined = []
for idx, row in fits_summary_01_wl.iterrows():
    
    row = fits_summary_01_wl.loc[0]

    hdul = fits.open(FITS_DIR / row['FILENAME'])
    sci = hdul[1].data
    flag = hdul[3].data # bitmap

    cutout_size = (51, 51)  # pixels
    cutout = Cutout2D(sci, (row['XCEN'], row['YCEN']), cutout_size, wcs=WCS(row['WCS_SERIALIZED']))
    flag_cutout = Cutout2D(flag, (row['XCEN'], row['YCEN']), cutout_size, wcs=WCS(row['WCS_SERIALIZED']))

    # xy center coordinates
    # xycen_obj_cutout = cutout.wcs.world_to_pixel(SkyCoord(ra=row['RA-OBJ'], dec=row['DEC-OBJ'], unit='deg'))
    # xycen_frag_cutout = cutout.wcs.world_to_pixel(SkyCoord(ra=row['RA-FRAG'], dec=row['DEC-FRAG'], unit='deg'))
    xycen_obj_cutout = cutout.to_cutout_position((row['XCEN'], row['YCEN']))
    xycen_frag_cutout = cutout.to_cutout_position((row['XCEN-FRAG'], row['YCEN-FRAG']))

    cutout_data_combined.append(cutout.data)

cutout_data_combined = np.array(cutout_data_combined)
cutout_data_combined = cutout_data_combined.mean(axis=0) # average cutout

contour_levels = np.nanpercentile(cutout_data_combined, [50, 70, 90])

fig, ax = plt.subplots(figsize=(6, 6))
interval = ZScaleInterval()
vmin, vmax = interval.get_limits(cutout_data_combined)
ax.imshow(cutout_data_combined, origin='lower', vmin=vmin, vmax=vmax, cmap='magma')
ax.contour(cutout_data_combined, levels=contour_levels, colors='white', linewidths=0.5)
ax.scatter(xycen_obj_cutout[0], xycen_obj_cutout[1], color='red', marker='X', s=200, label='Object Center')
ax.scatter(xycen_frag_cutout[0], xycen_frag_cutout[1], color='blue', marker='X', s=200, label='Fragment Center')
# ax.legend()
ax.set_title(f"{molecules} ({wl_min}-{wl_max} um, n={len(fits_summary_01_wl)})")
ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
plt.savefig(FIG_DIR / f"cutout_combined_01_{molecules}.png")
plt.show()

In [ ]:
molecules = "CO2"
wl_min, wl_max = dict_emission_um[molecules]
mask_wl = (fits_summary_01["WLEN-CEN"] >= wl_min) & (fits_summary_01["WLEN-CEN"] <= wl_max)
fits_summary_01_wl = fits_summary_01[mask_wl].reset_index(drop=True)
fits_summary_01_wl

In [ ]:
# cutout example
cutout_data_combined = []
for idx, row in fits_summary_01_wl.iterrows():
    
    row = fits_summary_01_wl.loc[0]

    hdul = fits.open(FITS_DIR / row['FILENAME'])
    sci = hdul[1].data
    flag = hdul[3].data # bitmap

    cutout_size = (51, 51)  # pixels
    cutout = Cutout2D(sci, (row['XCEN'], row['YCEN']), cutout_size, wcs=WCS(row['WCS_SERIALIZED']))
    flag_cutout = Cutout2D(flag, (row['XCEN'], row['YCEN']), cutout_size, wcs=WCS(row['WCS_SERIALIZED']))

    # xy center coordinates
    # xycen_obj_cutout = cutout.wcs.world_to_pixel(SkyCoord(ra=row['RA-OBJ'], dec=row['DEC-OBJ'], unit='deg'))
    # xycen_frag_cutout = cutout.wcs.world_to_pixel(SkyCoord(ra=row['RA-FRAG'], dec=row['DEC-FRAG'], unit='deg'))
    xycen_obj_cutout = cutout.to_cutout_position((row['XCEN'], row['YCEN']))
    xycen_frag_cutout = cutout.to_cutout_position((row['XCEN-FRAG'], row['YCEN-FRAG']))

    cutout_data_combined.append(cutout.data)

cutout_data_combined = np.array(cutout_data_combined)
cutout_data_combined = cutout_data_combined.mean(axis=0) # average cutout

contour_levels = np.percentile(cutout_data_combined, [50, 70, 90])

fig, ax = plt.subplots(figsize=(6, 6))
interval = ZScaleInterval()
vmin, vmax = interval.get_limits(cutout_data_combined)
ax.imshow(cutout_data_combined, origin='lower', vmin=vmin, vmax=vmax, cmap='magma')
ax.contour(cutout_data_combined, levels=contour_levels, colors='white', linewidths=0.5)
ax.scatter(xycen_obj_cutout[0], xycen_obj_cutout[1], color='red', marker='X', s=200, label='Object Center')
ax.scatter(xycen_frag_cutout[0], xycen_frag_cutout[1], color='blue', marker='X', s=200, label='Fragment Center')
# ax.legend()
ax.set_title(f"{molecules} ({wl_min}-{wl_max} um, n={len(fits_summary_01_wl)})")
ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
plt.savefig(FIG_DIR / f"cutout_combined_01_{molecules}.png")
plt.show()

In [ ]:
molecules = "Continuum"
wl_min, wl_max = dict_emission_um[molecules]
mask_wl = (fits_summary_02["WLEN-CEN"] >= wl_min) & (fits_summary_02["WLEN-CEN"] <= wl_max)
fits_summary_02_wl = fits_summary_02[mask_wl].reset_index(drop=True)
fits_summary_02_wl

In [ ]:
# cutout example
cutout_data_combined = []
for idx, row in fits_summary_02_wl.iterrows():
    
    row = fits_summary_02_wl.loc[0]

    hdul = fits.open(FITS_DIR / row['FILENAME'])
    sci = hdul[1].data
    flag = hdul[3].data # bitmap

    cutout_size = (51, 51)  # pixels
    cutout = Cutout2D(sci, (row['XCEN'], row['YCEN']), cutout_size, wcs=WCS(row['WCS_SERIALIZED']))
    flag_cutout = Cutout2D(flag, (row['XCEN'], row['YCEN']), cutout_size, wcs=WCS(row['WCS_SERIALIZED']))

    # xy center coordinates
    # xycen_obj_cutout = cutout.wcs.world_to_pixel(SkyCoord(ra=row['RA-OBJ'], dec=row['DEC-OBJ'], unit='deg'))
    # xycen_frag_cutout = cutout.wcs.world_to_pixel(SkyCoord(ra=row['RA-FRAG'], dec=row['DEC-FRAG'], unit='deg'))
    xycen_obj_cutout = cutout.to_cutout_position((row['XCEN'], row['YCEN']))
    xycen_frag_cutout = cutout.to_cutout_position((row['XCEN-FRAG'], row['YCEN-FRAG']))

    cutout_data_combined.append(cutout.data)

cutout_data_combined = np.array(cutout_data_combined)
cutout_data_combined = cutout_data_combined.mean(axis=0) # average cutout

contour_levels = np.percentile(cutout_data_combined, [50, 70, 90])

fig, ax = plt.subplots(figsize=(6, 6))
interval = ZScaleInterval()
vmin, vmax = interval.get_limits(cutout_data_combined)
ax.imshow(cutout_data_combined, origin='lower', vmin=vmin, vmax=vmax, cmap='magma')
ax.contour(cutout_data_combined, levels=contour_levels, colors='white', linewidths=0.5)
ax.scatter(xycen_obj_cutout[0], xycen_obj_cutout[1], color='red', marker='X', s=200, label='Object Center')
ax.scatter(xycen_frag_cutout[0], xycen_frag_cutout[1], color='blue', marker='X', s=200, label='Fragment Center')
# ax.legend()
ax.set_title(f"{molecules} ({wl_min}-{wl_max} um, n={len(fits_summary_02_wl)})")
ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
plt.savefig(FIG_DIR / f"cutout_combined_02_{molecules}.png")
plt.show()

In [ ]:
molecules = "H2O"
wl_min, wl_max = dict_emission_um[molecules]
mask_wl = (fits_summary_02["WLEN-CEN"] >= wl_min) & (fits_summary_02["WLEN-CEN"] <= wl_max)
fits_summary_02_wl = fits_summary_02[mask_wl].reset_index(drop=True)
fits_summary_02_wl

In [ ]:
# cutout example
cutout_data_combined = []
for idx, row in fits_summary_02_wl.iterrows():
    
    row = fits_summary_02_wl.loc[0]

    hdul = fits.open(FITS_DIR / row['FILENAME'])
    sci = hdul[1].data
    flag = hdul[3].data # bitmap

    cutout_size = (51, 51)  # pixels
    cutout = Cutout2D(sci, (row['XCEN'], row['YCEN']), cutout_size, wcs=WCS(row['WCS_SERIALIZED']))
    flag_cutout = Cutout2D(flag, (row['XCEN'], row['YCEN']), cutout_size, wcs=WCS(row['WCS_SERIALIZED']))

    # xy center coordinates
    # xycen_obj_cutout = cutout.wcs.world_to_pixel(SkyCoord(ra=row['RA-OBJ'], dec=row['DEC-OBJ'], unit='deg'))
    # xycen_frag_cutout = cutout.wcs.world_to_pixel(SkyCoord(ra=row['RA-FRAG'], dec=row['DEC-FRAG'], unit='deg'))
    xycen_obj_cutout = cutout.to_cutout_position((row['XCEN'], row['YCEN']))
    xycen_frag_cutout = cutout.to_cutout_position((row['XCEN-FRAG'], row['YCEN-FRAG']))

    cutout_data_combined.append(cutout.data)

cutout_data_combined = np.array(cutout_data_combined)
cutout_data_combined = cutout_data_combined.mean(axis=0) # average cutout

contour_levels = np.nanpercentile(cutout_data_combined, [50, 70, 90])

fig, ax = plt.subplots(figsize=(6, 6))
interval = ZScaleInterval()
vmin, vmax = interval.get_limits(cutout_data_combined)
ax.imshow(cutout_data_combined, origin='lower', vmin=vmin, vmax=vmax, cmap='magma')
ax.contour(cutout_data_combined, levels=contour_levels, colors='white', linewidths=0.5)
ax.scatter(xycen_obj_cutout[0], xycen_obj_cutout[1], color='red', marker='X', s=200, label='Object Center')
ax.scatter(xycen_frag_cutout[0], xycen_frag_cutout[1], color='blue', marker='X', s=200, label='Fragment Center')
# ax.legend()
ax.set_title(f"{molecules} ({wl_min}-{wl_max} um, n={len(fits_summary_02_wl)})")
ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
plt.savefig(FIG_DIR / f"cutout_combined_02_{molecules}.png")
plt.show()

In [ ]:
molecules = "CO2"
wl_min, wl_max = dict_emission_um[molecules]
mask_wl = (fits_summary_02["WLEN-CEN"] >= wl_min) & (fits_summary_02["WLEN-CEN"] <= wl_max)
fits_summary_02_wl = fits_summary_02[mask_wl].reset_index(drop=True)
fits_summary_02_wl

In [ ]:
# cutout example
cutout_data_combined = []
for idx, row in fits_summary_02_wl.iterrows():
    
    row = fits_summary_02_wl.loc[0]

    hdul = fits.open(FITS_DIR / row['FILENAME'])
    sci = hdul[1].data
    flag = hdul[3].data # bitmap

    cutout_size = (51, 51)  # pixels
    cutout = Cutout2D(sci, (row['XCEN'], row['YCEN']), cutout_size, wcs=WCS(row['WCS_SERIALIZED']))
    flag_cutout = Cutout2D(flag, (row['XCEN'], row['YCEN']), cutout_size, wcs=WCS(row['WCS_SERIALIZED']))

    # xy center coordinates
    # xycen_obj_cutout = cutout.wcs.world_to_pixel(SkyCoord(ra=row['RA-OBJ'], dec=row['DEC-OBJ'], unit='deg'))
    # xycen_frag_cutout = cutout.wcs.world_to_pixel(SkyCoord(ra=row['RA-FRAG'], dec=row['DEC-FRAG'], unit='deg'))
    xycen_obj_cutout = cutout.to_cutout_position((row['XCEN'], row['YCEN']))
    xycen_frag_cutout = cutout.to_cutout_position((row['XCEN-FRAG'], row['YCEN-FRAG']))

    cutout_data_combined.append(cutout.data)

cutout_data_combined = np.array(cutout_data_combined)
cutout_data_combined = cutout_data_combined.mean(axis=0) # average cutout

contour_levels = np.nanpercentile(cutout_data_combined, [50, 70, 90])

fig, ax = plt.subplots(figsize=(6, 6))
interval = ZScaleInterval()
vmin, vmax = interval.get_limits(cutout_data_combined)
ax.imshow(cutout_data_combined, origin='lower', vmin=vmin, vmax=vmax, cmap='magma')
ax.contour(cutout_data_combined, levels=contour_levels, colors='white', linewidths=0.5)
ax.scatter(xycen_obj_cutout[0], xycen_obj_cutout[1], color='red', marker='X', s=200, label='Object Center')
ax.scatter(xycen_frag_cutout[0], xycen_frag_cutout[1], color='blue', marker='X', s=200, label='Fragment Center')
# ax.legend()
ax.set_title(f"{molecules} ({wl_min}-{wl_max} um, n={len(fits_summary_02_wl)})")
ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
plt.savefig(FIG_DIR / f"cutout_combined_02_{molecules}.png")
plt.show()

In [ ]:
contour_levels